install all required packages


In [20]:


!pip install transformers[torch] datasets evaluate seqeval scikit-learn accelerate -q

print("✓ Libraries installed successfully")
print("✓ 'evaluate' library is now installed")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✓ Libraries installed successfully
✓ 'evaluate' library is now installed


In [21]:
import json
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')
import pickle   # <- standard library, always available
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Available: True
GPU Name: GRID V100S-32Q


In [22]:
# Define your custom entity types
# These are the categories you want the model to recognize

BASE_LABELS = [
    'CHANGE',       # transformation words (increased, decreased)
    'LOC',          # location names
    'LULC',         # Land Use/Land Cover (THIS IS YOUR MAIN FOCUS)
    'DATE',         # temporal references
    'PERCENT',      # percentage values
    'CARDINAL',     # numeric values
    'COORDINATES',  # geographic coordinates
    'SURFACE_UNIT', # area measurements
    'PROCESS',      # environmental processes
    'QUANTITY'      # other quantities
]

# Create IOB2 format labels (B- for Beginning, I- for Inside, O for Outside)
labels_list = ["O"]  # 'O' means the token is not part of any entity
for label in BASE_LABELS:
    labels_list.append(f"B-{label}")  # Beginning of entity
    labels_list.append(f"I-{label}")  # Inside/continuation of entity

# Create mappings between labels and IDs
label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

print(f"Total number of labels: {len(labels_list)}")
print(f"\nLabel to ID mapping (first 10):")
for label, id in list(label2id.items()):
    print(f"  {label}: {id}")

Total number of labels: 21

Label to ID mapping (first 10):
  O: 0
  B-CHANGE: 1
  I-CHANGE: 2
  B-LOC: 3
  I-LOC: 4
  B-LULC: 5
  I-LULC: 6
  B-DATE: 7
  I-DATE: 8
  B-PERCENT: 9
  I-PERCENT: 10
  B-CARDINAL: 11
  I-CARDINAL: 12
  B-COORDINATES: 13
  I-COORDINATES: 14
  B-SURFACE_UNIT: 15
  I-SURFACE_UNIT: 16
  B-PROCESS: 17
  I-PROCESS: 18
  B-QUANTITY: 19
  I-QUANTITY: 20


In [23]:
# You're missing some imports
from sklearn.model_selection import train_test_split  # Add this
import re  # For pattern matching

# Also, you reference 'tokenizer' before defining it
# Add this after your label definitions:
model_name = "roberta-base"  # or "roberta-large" for better performance
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
# Load your data file
# Change this path to your actual data file
DATA_FILE = 'merged_file2.json'  # CHANGE THIS TO YOUR FILE


with open(DATA_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} sentences from {DATA_FILE}")

# Explore the data structure
print("\nData structure example:")
print(f"Article ID: {raw_data[0]['article_id']}")
print(f"Text: {raw_data[4]['original_sentence']}")
print(f"\nEntities:")
for entity in raw_data[5]['entities']:
    print(f"  - '{entity['text']}' -> {entity['label']} (chars {entity['start_char']}-{entity['end_char']})")

# Count entities by type
entity_counts = defaultdict(int)
all_lulc_terms = []
article_counts = defaultdict(int)

for item in raw_data:
    article_counts[item['article_id']] += 1
    for entity in item['entities']:
        entity_counts[entity['label']] += 1
        
        # Collect LULC terms
        if entity['label'] == 'LULC':
            all_lulc_terms.append(entity['text'].lower())

print(f"\n📊 Dataset Statistics:")
print(f"Total sentences: {len(raw_data)}")
print(f"Total articles: {len(article_counts)}")
print(f"Average sentences per article: {len(raw_data) / len(article_counts):.1f}")

print("\n📈 Entity distribution in your data:")
for entity_type, count in sorted(entity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {entity_type}: {count}")

print(f"\n🏗️ LULC Statistics:")
print(f"Total LULC entities: {len(all_lulc_terms)}")
print(f"Unique LULC terms: {len(set(all_lulc_terms))}")
print("\nTop 10 most common LULC terms:")
from collections import Counter
lulc_counter = Counter(all_lulc_terms)
for term, count in lulc_counter.most_common(10):
    print(f"  • {term}: {count} occurrences")

Loaded 14522 sentences from merged_file2.json

Data structure example:
Article ID: Article_1
Text: The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.

Entities:
  - 'forest' -> LULC (chars 17-23)
  - 'declined' -> CHANGE (chars 30-38)
  - '15.25%' -> PERCENT (chars 52-58)
  - 'agriculture' -> LULC (chars 72-83)
  - '1.01%' -> PERCENT (chars 85-90)

📊 Dataset Statistics:
Total sentences: 14522
Total articles: 968
Average sentences per article: 15.0

📈 Entity distribution in your data:
  CHANGE: 10233
  LULC: 6424
  CARDINAL: 5419
  DATE: 4644
  LOC: 3868
  PERCENT: 1551
  COORDINATES: 1388
  PROCESS: 1241
  SURFACE_UNIT: 924
  QUANTITY: 130
  RESEARCH_TERM: 17

🏗️ LULC Statistics:
Total LULC entities: 6424
Unique LULC terms: 102

Top 10 most common LULC terms:
  • forest: 1267 occurrences
  • water: 798 occurrences
  • urban: 782 occurrences
  • forests: 389 occurrences
  • ci

In [7]:
def align_labels_with_tokens_enhanced(sentence_data, tokenizer):
    """
    Enhanced version that adds pattern-based LULC detection
    """
    text = sentence_data['original_sentence']
    entities = sentence_data['entities'][:]  # Copy existing entities
    
    # Find additional LULC patterns
    pattern_matches = find_lulc_patterns(text)
    
    # Remove overlaps: if a pattern overlaps with existing entity, 
    # prioritize LULC if the existing entity is not LULC
    for pattern_match in pattern_matches:
        overlaps = False
        for existing_entity in entities:
            # Check for overlap
            if (pattern_match['end'] > existing_entity['start_char'] and 
                pattern_match['start'] < existing_entity['end_char']):
                overlaps = True
                # If existing entity is not LULC, replace it
                if existing_entity['label'] != 'LULC':
                    print(f"Replacing {existing_entity['label']} '{existing_entity['text']}' with LULC '{pattern_match['text']}'")
                    existing_entity['start_char'] = pattern_match['start']
                    existing_entity['end_char'] = pattern_match['end']
                    existing_entity['text'] = pattern_match['text']
                    existing_entity['label'] = 'LULC'
                break
        
        # If no overlap, add as new entity
        if not overlaps:
            entities.append({
                'start_char': pattern_match['start'],
                'end_char': pattern_match['end'],
                'text': pattern_match['text'],
                'label': 'LULC'
            })
    
    # Sort entities by start position
    entities.sort(key=lambda x: x['start_char'])
    
    # Now proceed with original tokenization logic
    encoding = tokenizer(
        text,
        truncation=False,
        max_length=1024,
        return_offsets_mapping=True,
        padding=False
    )
    
    tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    offset_mapping = encoding['offset_mapping']
    
    # Initialize all labels as 'O'
    labels = ['O'] * len(tokens)
    
    # Process each entity (including newly added ones)
    for entity in entities:
        entity_start = entity['start_char']
        entity_end = entity['end_char']
        entity_label = entity['label']
        
        # Find tokens that overlap with this entity
        entity_tokens = []
        for idx, (token_start, token_end) in enumerate(offset_mapping):
            if token_start == 0 and token_end == 0:
                continue
                
            if token_end > entity_start and token_start < entity_end:
                entity_tokens.append(idx)
        
        # Assign labels
        if entity_tokens:
            labels[entity_tokens[0]] = f"B-{entity_label}"
            for idx in entity_tokens[1:]:
                labels[idx] = f"I-{entity_label}"
    
    return tokens, labels, offset_mapping

In [8]:
# Add this cell BEFORE your data processing
import re

def fix_missing_entities(data):
    """
    Automatically find and add missing LULC, LOC, and COORDINATES entities
    """
    print("🔧 Fixing missing entities...")
    
    fixed_data = []
    added_entities = 0
    
    for item in data:
        # Copy the original item
        fixed_item = {
            'original_sentence': item['original_sentence'],
            'entities': item['entities'][:],  # Copy existing entities
            'article_id': item.get('article_id', 'unknown')
        }
        
        text = item['original_sentence']
        
        # Get existing entity positions to avoid duplicates
        existing_spans = set()
        for entity in item['entities']:
            existing_spans.add((entity['start_char'], entity['end_char']))
        
        # PATTERN 1: Find "WORD area/land/zone" patterns for LULC
        lulc_patterns = [
            r'\b(forest\s+area)\b',           # "forest area" 
            r'\b(agricultural\s+land)\b',     # "agricultural land"
            r'\b(urban\s+areas?)\b',          # "urban area" or "urban areas"
            r'\b(crop\s*land)\b',             # "cropland" or "crop land"
            r'\b(grass\s*land)\b',            # "grassland" or "grass land"  
            r'\b(wet\s*lands?)\b',            # "wetland" or "wetlands"
            r'\b(residential\s+areas?)\b',    # "residential area"
            r'\b(commercial\s+areas?)\b',     # "commercial area"
            r'\b(industrial\s+areas?)\b',     # "industrial area"
            r'\b(urban\s+sprawl)\b',
            r'\b(built-?up\s+areas?)\b',
            r'\b(built\s+up\s+areas?)\b',
            r'\b(green\s+spaces?)\b'
        ]
        
        # PATTERN 2: Find "DIRECTION region/area/zone" patterns for LOC
        location_patterns = [
            r'\b(northern\s+region)\b',       # "northern region"
            r'\b(southern\s+region)\b',       # "southern region"  
            r'\b(eastern\s+region)\b',        # "eastern region"
            r'\b(western\s+region)\b',        # "western region"
            r'\b(central\s+region)\b',        # "central region"
            r'\b(coastal\s+areas?)\b',        # "coastal area"
            r'\b(rural\s+areas?)\b',          # "rural area"
            r'\b(mountain\s+regions?)\b',     # "mountain region"
        ]
        
        # PATTERN 3: Find coordinate patterns
        coordinate_patterns = [
            r'\b(\d+\.?\d*°[NSEW],?\s*\d+\.?\d*°[NSEW])\b',  # "23.5°N, 87.3°E"
            r'\b(\d+\.?\d*°[NSEW])\b',                        # "23.5°N"
        ]
        
        # Check each pattern type
        all_patterns = [
            (lulc_patterns, 'LULC'),
            (location_patterns, 'LOC'), 
            (coordinate_patterns, 'COORDINATES')
        ]
        
        for patterns, label_type in all_patterns:
            for pattern in patterns:
                # Find all matches in the text
                for match in re.finditer(pattern, text, re.IGNORECASE):
                    start_pos = match.start(1)  # Start of captured group
                    end_pos = match.end(1)     # End of captured group
                    matched_text = match.group(1)
                    
                    # Check if this position already has an entity
                    overlap = False
                    for existing_start, existing_end in existing_spans:
                        # Check if there's any overlap
                        if not (end_pos <= existing_start or start_pos >= existing_end):
                            overlap = True
                            break
                    
                    # If no overlap, add this as a new entity
                    if not overlap:
                        fixed_item['entities'].append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': label_type
                        })
                        existing_spans.add((start_pos, end_pos))
                        added_entities += 1
        
        fixed_data.append(fixed_item)
    
    print(f"✅ Added {added_entities} missing entities!")
    return fixed_data

# Test the function on a few examples first
test_sentences = [
    {
        'original_sentence': "The forest area decreased by 25% between 2010 and 2020 in the northern region.",
        'entities': [
            {'start_char': 4, 'end_char': 10, 'text': 'forest', 'label': 'LULC'},
            {'start_char': 16, 'end_char': 25, 'text': 'decreased', 'label': 'CHANGE'}
        ],
        'article_id': 'test1'
    },
    {
        'original_sentence': "Urban areas have replaced agricultural land at coordinates 23.5°N, 87.3°E.",
        'entities': [
            {'start_char': 0, 'end_char': 5, 'text': 'Urban', 'label': 'LULC'}
        ],
        'article_id': 'test2'
    },
    {
        'original_sentence': "The development of industrial parks replaced former wetlands and grasslands.",
        'entities': [
            {'start_char': 48, 'end_char': 56, 'text': 'wetlands', 'label': 'LULC'},
            {'start_char': 61, 'end_char': 71, 'text': 'grasslands', 'label': 'LULC'}
        ],
        'article_id': 'test2'
    }
]

print("🧪 Testing the fix function...")
fixed_test = fix_missing_entities(test_sentences)

print("\nBefore and after comparison:")
for i, (original, fixed) in enumerate(zip(test_sentences, fixed_test)):
    print(f"\nExample {i+1}: {original['original_sentence']}")
    print(f"  Original entities: {len(original['entities'])}")
    for entity in original['entities']:
        print(f"    - '{entity['text']}' → {entity['label']}")
    
    print(f"  Fixed entities: {len(fixed['entities'])}")
    for entity in fixed['entities']:
        print(f"    - '{entity['text']}' → {entity['label']}")

🧪 Testing the fix function...
🔧 Fixing missing entities...
✅ Added 3 missing entities!

Before and after comparison:

Example 1: The forest area decreased by 25% between 2010 and 2020 in the northern region.
  Original entities: 2
    - 'forest' → LULC
    - 'decreased' → CHANGE
  Fixed entities: 3
    - 'forest' → LULC
    - 'decreased' → CHANGE
    - 'northern region' → LOC

Example 2: Urban areas have replaced agricultural land at coordinates 23.5°N, 87.3°E.
  Original entities: 1
    - 'Urban' → LULC
  Fixed entities: 3
    - 'Urban' → LULC
    - 'agricultural land' → LULC
    - '23.5°N, 87.3°E' → COORDINATES

Example 3: The development of industrial parks replaced former wetlands and grasslands.
  Original entities: 2
    - 'wetlands' → LULC
    - 'grasslands' → LULC
  Fixed entities: 2
    - 'wetlands' → LULC
    - 'grasslands' → LULC


In [9]:
# Apply the fix to your actual data
print("🔧 Applying fixes to your training data...")

# IMPORTANT: Use the fixed data instead of raw_data
fixed_raw_data = fix_missing_entities(raw_data)

# Compare before and after
original_count = sum(len(item['entities']) for item in raw_data)
fixed_count = sum(len(item['entities']) for item in fixed_raw_data)

print(f"\n📊 Results:")
print(f"Original entities: {original_count}")
print(f"Fixed entities: {fixed_count}")
print(f"Added entities: {fixed_count - original_count}")

# Show some examples of what was added
print(f"\n🔍 Examples of added entities:")
examples_shown = 0
for original, fixed in zip(raw_data, fixed_raw_data):
    if len(fixed['entities']) > len(original['entities']) and examples_shown < 3:
        print(f"\nSentence: {original['original_sentence']}")
        original_entities = {(e['start_char'], e['end_char'], e['text']) for e in original['entities']}
        for entity in fixed['entities']:
            if (entity['start_char'], entity['end_char'], entity['text']) not in original_entities:
                print(f"  ➕ Added: '{entity['text']}' → {entity['label']}")
        examples_shown += 1

🔧 Applying fixes to your training data...
🔧 Fixing missing entities...
✅ Added 638 missing entities!

📊 Results:
Original entities: 35839
Fixed entities: 36477
Added entities: 638

🔍 Examples of added entities:

Sentence: The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.
  ➕ Added: 'built-up area' → LULC

Sentence: Using the outputs from remote sensing imagery, field surveys, and topped expert knowledge of the study area, five LULC types were classified; water body, bare ground, built-up area, forest, and agriculture ( Fig.
  ➕ Added: 'built-up area' → LULC

Sentence: The resultant improvement in basic facilities combined with creation of additional jobs further attracted more immigration from rural areas and lesser developed cities, causing a boom in real estate and informal settlements and triggering significant changes in land uses.
  ➕ Added: 'rural areas' → LOC


In [11]:
def fix_buildup_labeling_consistency(fixed_raw_data):
    """
    Fix the inconsistent labeling of 'built-up area' patterns
    """
    print("🔧 FIXING BUILT-UP AREA LABELING CONSISTENCY...")
    
    import re
    fixed_count = 0
    sentences_fixed = 0
    
    # Patterns to fix
    buildup_patterns = [
        r'\b(built-up\s+areas?)\b',
        r'\b(built\s+up\s+areas?)\b',
        r'\b(build-up\s+areas?)\b'
    ]
    
    for item in raw_data:
        text = item['original_sentence']
        entities = item['entities']
        
        sentence_had_buildup = False
        
        # Check if sentence contains built-up area patterns
        for pattern in buildup_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                sentence_had_buildup = True
                start_pos = match.start(1)
                end_pos = match.end(1)
                matched_text = match.group(1)
                
                # Check if this span is already labeled as LULC
                already_labeled_lulc = any(
                    entity['start_char'] <= start_pos < end_pos <= entity['end_char'] and 
                    entity['label'] == 'LULC'
                    for entity in entities
                )
                
                if not already_labeled_lulc:
                    # Check if it overlaps with any existing entity
                    overlaps = any(
                        not (end_pos <= entity['start_char'] or start_pos >= entity['end_char'])
                        for entity in entities
                    )
                    
                    if not overlaps:
                        # Add as LULC entity
                        item['entities'].append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': 'LULC'
                        })
                        fixed_count += 1
                        print(f"  ➕ Fixed: '{matched_text}' in sentence: {text[:50]}...")
                    else:
                        # Remove conflicting entities and add LULC
                        original_count = len(entities)
                        item['entities'] = [
                            entity for entity in entities
                            if (end_pos <= entity['start_char'] or start_pos >= entity['end_char'])
                        ]
                        removed_count = original_count - len(item['entities'])
                        
                        item['entities'].append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': 'LULC'
                        })
                        fixed_count += 1
                        print(f"  🔄 Replaced {removed_count} entities with LULC: '{matched_text}'")
        
        if sentence_had_buildup:
            sentences_fixed += 1
    
    print(f"\n✅ CONSISTENCY FIX COMPLETE:")
    print(f"   Fixed entities: {fixed_count}")
    print(f"   Sentences affected: {sentences_fixed}")
    
    return raw_data

# Apply the fix
print("Applying built-up area labeling consistency fix...")
fixed_raw_data = fix_buildup_labeling_consistency(raw_data.copy())

Applying built-up area labeling consistency fix...
🔧 FIXING BUILT-UP AREA LABELING CONSISTENCY...

✅ CONSISTENCY FIX COMPLETE:
   Fixed entities: 0
   Sentences affected: 256


In [12]:
def fix_enhanced_location_patterns(fixed_raw_data):
    """
    Enhanced location pattern detection for complex directional terms
    """
    print("🔧 FIXING ENHANCED LOCATION PATTERNS...")
    
    import re
    fixed_count = 0
    
    # Enhanced location patterns - more comprehensive
    enhanced_location_patterns = [
        # Basic directional regions
        r'\b(northern\s+region)\b',
        r'\b(southern\s+region)\b',
        r'\b(eastern\s+region)\b', 
        r'\b(western\s+region)\b',
        r'\b(central\s+region)\b',
        
        # Compound directional regions (the missing ones!)
        r'\b(northeastern\s+region)\b',
        r'\b(northwestern\s+region)\b', 
        r'\b(southeastern\s+region)\b',
        r'\b(southwestern\s+region)\b',
        r'\b(north-eastern\s+region)\b',
        r'\b(north-western\s+region)\b',
        r'\b(south-eastern\s+region)\b', 
        r'\b(south-western\s+region)\b',
        
        # Regional qualifiers with country/place names
        r'\b(northeastern\s+region\s+of\s+\w+)\b',
        r'\b(northwestern\s+region\s+of\s+\w+)\b',
        r'\b(southeastern\s+region\s+of\s+\w+)\b',
        r'\b(southwestern\s+region\s+of\s+\w+)\b',
        r'\b(northern\s+region\s+of\s+\w+)\b',
        r'\b(southern\s+region\s+of\s+\w+)\b',
        r'\b(eastern\s+region\s+of\s+\w+)\b',
        r'\b(western\s+region\s+of\s+\w+)\b',
        r'\b(central\s+region\s+of\s+\w+)\b',
        
        # Other location types
        r'\b(coastal\s+areas?)\b',
    ]
    
    for item in raw_data:
        text = item['original_sentence']
        entities = item['entities']
        existing_spans = {(e['start_char'], e['end_char']) for e in entities}
        
        for pattern in enhanced_location_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                start_pos = match.start(1)
                end_pos = match.end(1) 
                matched_text = match.group(1)
                
                # Check if already labeled as LOC
                already_labeled_loc = any(
                    entity['start_char'] <= start_pos < end_pos <= entity['end_char'] and 
                    entity['label'] == 'LOC'
                    for entity in entities
                )
                
                if not already_labeled_loc:
                    # Check for overlaps
                    overlaps = any(
                        not (end_pos <= e_start or start_pos >= e_end)
                        for e_start, e_end in existing_spans
                    )
                    
                    if not overlaps:
                        # Add as LOC entity
                        item['entities'].append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': 'LOC'
                        })
                        existing_spans.add((start_pos, end_pos))
                        fixed_count += 1
                        print(f"  ➕ Fixed: '{matched_text}' as LOC")
    
    print(f"\n✅ Enhanced location patterns fixed: {fixed_count} entities")
    return fixed_raw_data

# Apply enhanced location fix
enhanced_fixed_data = fix_enhanced_location_patterns(fixed_raw_data.copy())

🔧 FIXING ENHANCED LOCATION PATTERNS...
  ➕ Fixed: 'southeastern region' as LOC
  ➕ Fixed: 'coastal area' as LOC
  ➕ Fixed: 'southeastern region' as LOC
  ➕ Fixed: 'northern region' as LOC
  ➕ Fixed: 'northern region' as LOC
  ➕ Fixed: 'eastern region' as LOC
  ➕ Fixed: 'western region' as LOC
  ➕ Fixed: 'northern region' as LOC
  ➕ Fixed: 'Northeastern region' as LOC
  ➕ Fixed: 'northwestern region' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal area' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'northern region' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal area' as LOC
  ➕ Fixed: 'northern region' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal area' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'coastal areas' as LOC
  ➕ Fixed: 'Northeastern region' as LOC
  ➕ F

In [13]:
def fix_surface_unit_labeling(fixed_raw_data):
    """
    Fix missing SURFACE_UNIT labels for area measurements
    Following the same pattern as other fix functions
    """
    print("🔧 FIXING SURFACE_UNIT LABELING...")
    
    import re
    fixed_count = 0
    sentences_fixed = 0
    
    # Surface unit patterns
    surface_unit_patterns = [
        r'\b(hectares?)\b',                    # hectares, hectare
        r'\b(km²|km2)\b',                      # km², km2  
        r'\b(square\s+kilometers?)\b',         # square kilometer(s)
        r'\b(sq\.?\s*km)\b',                   # sq km, sq. km
        r'\b(m²|m2)\b',                        # m², m2
        r'\b(square\s+meters?)\b',             # square meter(s)
        r'\b(acres?)\b',                       # acres, acre
        r'\b(ha)\b',                           # ha (hectare abbreviation)
        r'\b(square\s+miles?)\b',              # square mile(s)
    
        # Number + hectares/ha patterns
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s*(?:hectares?|ha))\b',           # 20,385ha, 1,500 hectares
        r'\b(\d+(?:\.\d+)?\s*(?:million|billion)\s+hectares?)\b',       # 1.5 million hectares
        
        # Number + km² patterns  
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s*(?:km²|km2))\b',               # 500km², 1,200 km²
        r'\b(\d+(?:\.\d+)?\s*(?:million|billion)\s+km²)\b',            # 2.3 million km²
        
        # Number + square kilometers
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s+square\s+kilometers?)\b',      # 500 square kilometers
        r'\b(\d+(?:\.\d+)?\s*(?:million|billion)\s+square\s+kilometers?)\b', # 1.5 million square kilometers
        
        # Number + acres
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s*acres?)\b',                    # 1,500acres, 200 acres
        r'\b(\d+(?:\.\d+)?\s*(?:million|billion)\s+acres?)\b',         # 2.5 million acres
        
        # Number + square meters
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s*(?:m²|m2))\b',                 # 5,000m²
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s+square\s+meters?)\b',          # 1,000 square meters
        
        # Number + square miles
        r'\b(\d+(?:,\d{3})*(?:\.\d+)?\s*(?:square\s+miles?|sq\.?\s*mi))\b', # 100 square miles
        
        # Just units (when number is separate)
        r'\b(hectares?)\b',                    # hectares, hectare (standalone)
        r'\b(km²|km2)\b',                      # km², km2 (standalone)
        r'\b(acres?)\b',                       # acres, acre (standalone)
        r'\b(ha)\b',                           # ha (standalone)
    ]
    
    for item in fixed_raw_data:
        text = item['original_sentence']
        entities = item['entities']
        
        sentence_had_surface_unit = False
        
        # Check if sentence contains surface unit patterns
        for pattern in surface_unit_patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                sentence_had_surface_unit = True
                start_pos = match.start(1)
                end_pos = match.end(1)
                matched_text = match.group(1)
                
                # Check if this span is already labeled as SURFACE_UNIT
                already_labeled_surface_unit = any(
                    entity['start_char'] <= start_pos < end_pos <= entity['end_char'] and 
                    entity['label'] == 'SURFACE_UNIT'
                    for entity in entities
                )
                
                if not already_labeled_surface_unit:
                    # Check if it overlaps with any existing entity
                    overlaps = any(
                        not (end_pos <= entity['start_char'] or start_pos >= entity['end_char'])
                        for entity in entities
                    )
                    
                    if not overlaps:
                        # Add as SURFACE_UNIT entity
                        entities.append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': 'SURFACE_UNIT'
                        })
                        fixed_count += 1
                        print(f"  ➕ Fixed: '{matched_text}' in sentence: {text[:50]}...")
                    else:
                        # Remove conflicting entities and add SURFACE_UNIT
                        original_count = len(entities)
                        entities[:] = [
                            entity for entity in entities
                            if (end_pos <= entity['start_char'] or start_pos >= entity['end_char'])
                        ]
                        
                        entities.append({
                            'start_char': start_pos,
                            'end_char': end_pos,
                            'text': matched_text,
                            'label': 'SURFACE_UNIT'
                        })
                        fixed_count += 1
                        print(f"  🔄 Replaced entities with SURFACE_UNIT: '{matched_text}'")
        
        if sentence_had_surface_unit:
            sentences_fixed += 1
    
    print(f"\n✅ SURFACE_UNIT FIX COMPLETE:")
    print(f"   Fixed entities: {fixed_count}")
    print(f"   Sentences affected: {sentences_fixed}")
    
    return fixed_raw_data

# Apply the fix to your data (following the same pattern as your other fixes)
print("Applying SURFACE_UNIT fixes to your training data...")

# Apply the fix - this modifies your data directly like the other functions
fixed_raw_data = fix_surface_unit_labeling(fixed_raw_data.copy())

Applying SURFACE_UNIT fixes to your training data...
🔧 FIXING SURFACE_UNIT LABELING...
  ➕ Fixed: 'hectares' in sentence: It is also defined as urban expansion, as the proc...
  ➕ Fixed: 'hectares' in sentence: It is also defined as urban expansion, as the proc...
  ➕ Fixed: 'hectares' in sentence: It is also defined as urban expansion, as the proc...
  🔄 Replaced entities with SURFACE_UNIT: '13 billion hectares'
  🔄 Replaced entities with SURFACE_UNIT: '4.9 billion hectares'
  🔄 Replaced entities with SURFACE_UNIT: '420 million hectares'
  ➕ Fixed: 'hectares' in sentence: As for urban land, this area could expand from 40 ...
  🔄 Replaced entities with SURFACE_UNIT: '143 million hectares'
  ➕ Fixed: 'hectares' in sentence: In 2011-2015, following the data from MoNRE, Vietn...
  ➕ Fixed: 'hectares' in sentence: Economically, urbanization and industrialization h...
  🔄 Replaced entities with SURFACE_UNIT: '11 million hectares'
  ➕ Fixed: 'hectare' in sentence: From an organism's perspect

In [13]:
def prepare_dataset(data, tokenizer, label2id, max_length=512):
    def process_example(example):
        sentence = example['original_sentence']
        entities = example['entities']
        
        # Tokenize the sentence
        encoding = tokenizer(
            sentence, 
            truncation=True, 
            max_length=max_length, 
            return_offsets_mapping=True, 
            padding='max_length'
        )
        
        offset_mapping = encoding['offset_mapping']
        labels = ['O'] * len(offset_mapping)
        
        # Process each entity
        for entity in entities:
            start_char = entity['start_char']
            end_char = entity['end_char']
            entity_label = entity['label']
            
            # Skip if label not in our label set
            if entity_label not in BASE_LABELS:
                continue
            
            # Find tokens that belong to this entity
            entity_tokens = []
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                # Skip special tokens
                if token_start == 0 and token_end == 0:
                    continue
                    
                # Check if token overlaps with entity
                if token_start < end_char and token_end > start_char:
                    entity_tokens.append(idx)
            
            # Assign labels
            if entity_tokens:
                labels[entity_tokens[0]] = f'B-{entity_label}'
                for idx in entity_tokens[1:]:
                    labels[idx] = f'I-{entity_label}'
        
        # Convert labels to IDs
        label_ids = []
        for label in labels:
            if label in label2id:
                label_ids.append(label2id[label])
            else:
                label_ids.append(label2id['O'])
        
        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': label_ids,
            'original_sentence': sentence
        }
    
    processed_data = []
    for i, example in enumerate(data):
        if i % 1000 == 0:
            print(f"🔄 Processed {i}/{len(data)}")
        processed_data.append(process_example(example))
    return processed_data

# Process the data
processed_data = prepare_dataset(fixed_raw_data, tokenizer, label2id)
# Split the data
print("\n✂️ Splitting data into train/test sets...")
train_data, test_data = train_test_split(
    processed_data, 
    test_size=0.1, 
    random_state=42, 
    shuffle=True
)

print(f"📊 Train set: {len(train_data)} examples")
print(f"📊 Test set: {len(test_data)} examples")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("\n✅ Dataset ready for training!")

# Analyze entity distribution in train set
def analyze_entity_distribution(dataset, id2label):
    entity_counts = defaultdict(int)
    total_entities = 0
    
    for example in dataset:
        for label_id in example['labels']:
            label = id2label[label_id]
            if label != 'O':
                if label.startswith('B-'):
                    entity_type = label[2:]
                    entity_counts[entity_type] += 1
                    total_entities += 1
    
    print("\n📊 Entity distribution in training set:")
    for entity_type, count in sorted(entity_counts.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total_entities) * 100 if total_entities > 0 else 0
        print(f"  {entity_type}: {count} ({percentage:.1f}%)")
        if entity_type == 'LULC':
            print(f"    ⭐ This is your main focus!")

analyze_entity_distribution(train_dataset, id2label)

# Save everything
dataset_dict.save_to_disk('./processed_dataset')
print("\n💾 Dataset saved to './processed_dataset'")

with open('label_mappings.pkl', 'wb') as f:
    pickle.dump({'label2id': label2id, 'id2label': id2label}, f)
print("✅ Label mappings saved to 'label_mappings.pkl'")

🔄 Processed 0/14522
🔄 Processed 1000/14522
🔄 Processed 2000/14522
🔄 Processed 3000/14522
🔄 Processed 4000/14522
🔄 Processed 5000/14522
🔄 Processed 6000/14522
🔄 Processed 7000/14522
🔄 Processed 8000/14522
🔄 Processed 9000/14522
🔄 Processed 10000/14522
🔄 Processed 11000/14522
🔄 Processed 12000/14522
🔄 Processed 13000/14522
🔄 Processed 14000/14522

✂️ Splitting data into train/test sets...
📊 Train set: 13069 examples
📊 Test set: 1453 examples

✅ Dataset ready for training!

📊 Entity distribution in training set:
  CHANGE: 9123 (28.2%)
  LULC: 5994 (18.5%)
    ⭐ This is your main focus!
  CARDINAL: 4878 (15.1%)
  DATE: 4178 (12.9%)
  LOC: 3493 (10.8%)
  PERCENT: 1391 (4.3%)
  COORDINATES: 1161 (3.6%)
  PROCESS: 1134 (3.5%)
  SURFACE_UNIT: 919 (2.8%)
  QUANTITY: 112 (0.3%)


Saving the dataset (0/1 shards):   0%|          | 0/13069 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1453 [00:00<?, ? examples/s]


💾 Dataset saved to './processed_dataset'
✅ Label mappings saved to 'label_mappings.pkl'


In [14]:
# Load pre-trained RoBERTa and configure it for token classification
model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Move model to GPU if available
model = model.to(device)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Number of labels: {model.config.num_labels}")

# This is what makes RoBERTa context-aware:
# - It has 12 transformer layers that process the entire sentence
# - Each token's representation is influenced by all other tokens
# - The self-attention mechanism learns which tokens to focus on

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: roberta-base
Number of parameters: 124,071,189
Number of labels: 21


In [15]:
# Load the seqeval metric for NER evaluation
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """
    Compute precision, recall, and F1 score for each entity type.
    This helps you understand how well the model performs on each category.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Convert predictions to label strings
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    
    # Extract per-entity metrics
    per_entity_results = {}
    for entity in BASE_LABELS:
        if entity in results:
            per_entity_results[f"{entity}_f1"] = results[entity]["f1"]
            per_entity_results[f"{entity}_precision"] = results[entity]["precision"]
            per_entity_results[f"{entity}_recall"] = results[entity]["recall"]
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
        **per_entity_results
    }

print("Evaluation metrics configured.")
print("Will track performance for each entity type, especially LULC.")

Evaluation metrics configured.
Will track performance for each entity type, especially LULC.


In [16]:
# Training configuration
# These parameters control how the model learns

training_args = TrainingArguments(
    output_dir="./roberta_ner_model",           # Where to save the model
    learning_rate=3e-5,                         # How fast the model learns (don't set too high!)
    per_device_train_batch_size=8,              # Samples per batch (reduce if GPU memory issues)
    per_device_eval_batch_size=8,               
    num_train_epochs=6,                         # Number of complete passes through data
    weight_decay=0.01,                          # Regularization to prevent overfitting
    evaluation_strategy="epoch",                # When to evaluate
    save_strategy="epoch",                      # When to save checkpoints
    logging_steps=50,                           # How often to log progress
    load_best_model_at_end=True,               # Keep the best model
    metric_for_best_model="f1",                # What metric to use for "best"
    push_to_hub=False,                         # Don't upload to Hugging Face Hub
    report_to="none",                          # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),            # Use mixed precision if GPU available
    lr_scheduler_type = "linear",
    seed= 42
)

# Data collator handles batching and padding
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print(f"Training configuration set:")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Mixed precision training: {training_args.fp16}")

Training configuration set:
  - Epochs: 6
  - Batch size: 8
  - Learning rate: 3e-05
  - Mixed precision training: True


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],  # <-- Changed from dataset_split
    eval_dataset=dataset_dict["test"],    # <-- Changed from dataset_split
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized and ready.")
print(f"Will train for {len(dataset_dict['train']) // training_args.per_device_train_batch_size * training_args.num_train_epochs} steps")  # <-- Updated to use dataset_dict

Trainer initialized and ready.
Will train for 9798 steps


In [18]:
# Start training!
# This will take some time depending on your hardware
# With GPU: ~5-10 minutes for 2000 sentences
# With CPU: ~30-60 minutes
import time

print("Starting training... This will take some time.")
print("You'll see progress updates and evaluation results after each epoch.\n")

# Train the model
start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time


# Save the final model
trainer.save_model("./final_ner_model")
tokenizer.save_pretrained("./final_ner_model")
print(f"⏱️ Training time: {training_time/60:.1f} minutes")
print("\n✓ Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")

Starting training... This will take some time.
You'll see progress updates and evaluation results after each epoch.



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Change F1,Change Precision,Change Recall,Loc F1,Loc Precision,Loc Recall,Lulc F1,Lulc Precision,Lulc Recall,Date F1,Date Precision,Date Recall,Percent F1,Percent Precision,Percent Recall,Cardinal F1,Cardinal Precision,Cardinal Recall,Coordinates F1,Coordinates Precision,Coordinates Recall,Surface Unit F1,Surface Unit Precision,Surface Unit Recall,Process F1,Process Precision,Process Recall,Quantity F1,Quantity Precision,Quantity Recall
1,0.006700,0.006584,0.840184,0.865369,0.852591,0.998017,0.979409,0.971581,0.987365,0.693023,0.669663,0.718072,0.974582,0.973875,0.975291,0.830031,0.805668,0.855914,0.808383,0.771429,0.849057,0.762726,0.697161,0.841905,0.432927,0.617391,0.333333,0.649573,0.593750,0.716981,0.986047,0.981481,0.990654,0.000000,0.000000,0.000000
2,0.004000,0.005551,0.821606,0.871943,0.846026,0.998266,0.983871,0.976868,0.990975,0.661728,0.678481,0.645783,0.978970,0.976845,0.981105,0.835391,0.800789,0.873118,0.834320,0.787709,0.886792,0.796594,0.791353,0.801905,0.387681,0.315634,0.502347,0.726531,0.640288,0.839623,0.981481,0.972477,0.990654,0.263158,0.238095,0.294118
3,0.003200,0.005166,0.872379,0.896923,0.884481,0.998412,0.989247,0.982206,0.996390,0.741082,0.709251,0.775904,0.978386,0.970000,0.986919,0.800834,0.777328,0.825806,0.870370,0.854545,0.886792,0.842105,0.780488,0.914286,0.588235,0.787402,0.469484,0.840183,0.814159,0.867925,0.990741,0.981651,1.000000,0.153846,0.222222,0.117647
4,0.002200,0.005492,0.839329,0.907967,0.872300,0.998400,0.986179,0.974449,0.998195,0.739130,0.703704,0.778313,0.972662,0.962963,0.982558,0.861570,0.829026,0.896774,0.870370,0.854545,0.886792,0.836717,0.779605,0.902857,0.420233,0.358804,0.507042,0.872247,0.818182,0.933962,0.986175,0.972727,1.000000,0.148148,0.200000,0.117647
5,0.001300,0.005100,0.893285,0.902445,0.897842,0.998636,0.987025,0.978705,0.995487,0.789598,0.774942,0.804819,0.984104,0.978448,0.989826,0.852008,0.837838,0.866667,0.873846,0.855422,0.893082,0.823204,0.796791,0.851429,0.636103,0.816176,0.521127,0.883929,0.838983,0.933962,0.990741,0.981651,1.000000,0.294118,0.294118,0.294118
6,0.001200,0.005293,0.892109,0.906653,0.899322,0.998649,0.988804,0.981333,0.996390,0.791027,0.775463,0.807229,0.987672,0.985528,0.989826,0.858044,0.839506,0.877419,0.864198,0.848485,0.880503,0.834862,0.805310,0.866667,0.612466,0.724359,0.530516,0.880000,0.831933,0.933962,0.990741,0.981651,1.000000,0.352941,0.352941,0.352941


⏱️ Training time: 18.2 minutes

✓ Training completed!
Training loss: 0.0067


In [20]:
# Evaluate the model on the test set
print("Evaluating model performance...\n")
start_time = time.time()

eval_results = trainer.evaluate()
training_time = time.time() - start_time

# Display overall metrics
print("Overall Performance:")
print(f"  Precision: {eval_results['eval_precision']:.3f}")
print(f"  Recall: {eval_results['eval_recall']:.3f}")
print(f"  F1 Score: {eval_results['eval_f1']:.3f}")
print(f"  Accuracy: {eval_results['eval_accuracy']:.3f}")

# Display per-entity metrics if available
print("\nPer-Entity Performance:")
for entity in BASE_LABELS:
    f1_key = f"eval_{entity}_f1"
    if f1_key in eval_results:
        print(f"  {entity}:")
        print(f"    F1: {eval_results[f1_key]:.3f}")
        print(f"    Precision: {eval_results.get(f'eval_{entity}_precision', 0):.3f}")
        print(f"    Recall: {eval_results.get(f'eval_{entity}_recall', 0):.3f}")

Evaluating model performance...

Overall Performance:
  Precision: 0.892
  Recall: 0.907
  F1 Score: 0.899
  Accuracy: 0.999

Per-Entity Performance:
  CHANGE:
    F1: 0.989
    Precision: 0.981
    Recall: 0.996
  LOC:
    F1: 0.791
    Precision: 0.775
    Recall: 0.807
  LULC:
    F1: 0.988
    Precision: 0.986
    Recall: 0.990
  DATE:
    F1: 0.858
    Precision: 0.840
    Recall: 0.877
  PERCENT:
    F1: 0.864
    Precision: 0.848
    Recall: 0.881
  CARDINAL:
    F1: 0.835
    Precision: 0.805
    Recall: 0.867
  COORDINATES:
    F1: 0.612
    Precision: 0.724
    Recall: 0.531
  SURFACE_UNIT:
    F1: 0.880
    Precision: 0.832
    Recall: 0.934
  PROCESS:
    F1: 0.991
    Precision: 0.982
    Recall: 1.000
  QUANTITY:
    F1: 0.353
    Precision: 0.353
    Recall: 0.353


In [21]:
def predict_entities(text, model, tokenizer):
    """Test the model with new text"""
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    )
    
    # Move to device
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    offset_mapping = inputs["offset_mapping"][0].tolist()
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Convert predictions to labels
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    predicted_labels = [id2label[pred.item()] for pred in predictions[0]]
    
    # Extract entities
    entities = []
    current_entity = None
    
    for idx, (token, label, (start, end)) in enumerate(zip(tokens, predicted_labels, offset_mapping)):
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "text": tokenizer.decode(input_ids[0][idx:idx+1]),
                "label": label[2:],
                "start": start,
                "end": end,
                "tokens": [token]
            }
        elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
            current_entity["tokens"].append(token)
            current_entity["end"] = end
            current_entity["text"] = text[current_entity["start"]:current_entity["end"]]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    return entities

# Test with examples
test_sentences = [
    "In the Amazon Basin (~3°S, 60°W), between 2001 and 2020, forest cover decreased by ~9%, representing a loss of over 54.2 million hectares, largely due to deforestation.",
    "Globally, approximately 420 million hectares of forest have been lost since 1990 (~10% of total forest area), mainly to agriculture, though the annual deforestation rate fell from ~16 million ha in the 1990s to ~10 million ha in the late 2010s.",
    "The Sahara Desert (~23°N, 13°E) has expanded by about 10% since 1920, encroaching on an additional ~700,000 km² of land due to climate change and reduced rainfall (accelerating desertification).",
    "In the Gran Chaco (~24°S, 62°W), over 20% of the native forest has been cleared since the 1980s (~142,000 km²) for cattle ranching and soybean farming, making it one of the world's most rapidly deforested regions.",
    "In Indonesia (~1°S, 113°E), rapid deforestation caused forest cover to shrink by ~20% between 1990 and 2010 (a loss of around 24 million hectares), primarily due to the expansion of oil palm plantations.",
    "In China (~35°N, 105°E), urban land area increased by nearly 500% between 1992 and 2015 (from ~12,200 km² to ~72,900 km²) due to rapid urbanization and economic development, often replacing agricultural land.",
    "In China (~35°N, 110°E), forest cover rose from 16.7% to 22.5% of the country's area between 1990 and 2015 – a gain of about 511,800 km² – thanks to large-scale afforestation programs.",
    "In the European Union (~50°N, 15°E), forest area expanded by nearly 10% from 1990 to 2020, growing from ~145 million to 159 million hectares, as farmland abandonment and reforestation projects increased wooded land.",
    "In Russia (~55°N, 40°E), around 50 million hectares of cropland (about 34% of the 1991 farmland area) were abandoned after the Soviet Union's collapse, allowing forests and grasslands to regenerate naturally and sequester ~50 million tons of carbon annually.",
    "In Saudi Arabia (~24°N, 45°E), desert areas were converted to farmland through intensive irrigation; irrigated cropland expanded from under 4,000 km² in 1970 to over 8,000 km² by the early 1990s (>100% increase), although this led to >60% depletion of nonrenewable groundwater over 25 years.",
    "In Central Asia (~45°N, 60°E), the Aral Sea has largely dried up: its surface area shrank by ~85% from 1960 to 2012 (from ~67,500 km² to ~10,000 km²) as Soviet river diversions for irrigation created a new desert on the former lakebed.",
    "In West Africa (~13°N, 14°E), Lake Chad's surface area has plummeted by over 90% since the 1960s (shrinking from ~25,000 km² to <2,000 km² by the early 2000s) due to prolonged drought and water diversion, devastating local agriculture and fisheries.",
    "In Vietnam’s Mekong Delta (~10°N, 106°E), mangrove forests declined by over 50% from the late 1970s to early 1990s, as ~100,000 hectares were cleared for shrimp aquaculture and other development.",
    "In India (~22°N, 79°E), the area of land undergoing degradation increased from ~94.5 million hectares (28.8% of the country) in 2005 to ~96.4 million hectares (29.3%) by 2013, reflecting growing desertification caused by deforestation, overgrazing, and soil erosion.",


]
print("\n🧪 Testing model with example sentences:")
for sentence in test_sentences:
    print(f"\n📝 Input: {sentence}")
    entities = predict_entities(sentence, model, tokenizer)
    print("🏷️ Detected entities:")
    for entity in entities:
        print(f"   - {entity['text']} → {entity['label']}")


🧪 Testing model with example sentences:

📝 Input: In the Amazon Basin (~3°S, 60°W), between 2001 and 2020, forest cover decreased by ~9%, representing a loss of over 54.2 million hectares, largely due to deforestation.
🏷️ Detected entities:
   -  60 → CARDINAL
   - between 2001 and 2020 → DATE
   -  forest → LULC
   -  decreased → CHANGE
   - ~9%, → PERCENT
   -  loss → CHANGE
   - 54.2 million hectares → SURFACE_UNIT
   -  deforestation → PROCESS

📝 Input: Globally, approximately 420 million hectares of forest have been lost since 1990 (~10% of total forest area), mainly to agriculture, though the annual deforestation rate fell from ~16 million ha in the 1990s to ~10 million ha in the late 2010s.
🏷️ Detected entities:
   - 420 million hectares → SURFACE_UNIT
   -  forest → LULC
   -  1990 → DATE
   -  forest → LULC
   -  agriculture → LULC
   -  annual → DATE
   -  deforestation → PROCESS
   -  ha → SURFACE_UNIT
   - the 1990s → DATE
   -  ha → SURFACE_UNIT
   - the late 2010s → DATE

In [24]:
def extract_entities(text, model_path="./final_ner_model"):
    """
    Extract entities from new text using the trained model.
    This function shows how to use your model in practice.
    """
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.eval()
    
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
        padding=True
    )
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**{k: v for k, v in inputs.items() if k != 'offset_mapping'})
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Convert predictions to labels
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    labels = [model.config.id2label[pred.item()] for pred in predictions[0]]
    offset_mapping = inputs['offset_mapping'][0]
    
    # Extract entities
    entities = []
    current_entity = None
    
    for idx, (token, label, offset) in enumerate(zip(tokens, labels, offset_mapping)):
        if token in tokenizer.all_special_tokens:
            continue
            
        if label.startswith('B-'):
            # Start of new entity
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'text': tokenizer.convert_tokens_to_string([token]),
                'label': label[2:],
                'start': offset[0].item(),
                'end': offset[1].item(),
                'tokens': [token]
            }
        elif label.startswith('I-') and current_entity and label[2:] == current_entity['label']:
            # Continuation of current entity
            current_entity['tokens'].append(token)
            current_entity['end'] = offset[1].item()
            current_entity['text'] = tokenizer.convert_tokens_to_string(current_entity['tokens'])
        else:
            # Not an entity
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    # Clean up entity text
    for entity in entities:
        entity['text'] = entity['text'].strip()
        del entity['tokens']
    
    return entities

# Test the extraction function
test_sentences = [
    "In the Congo Basin (~0°, 20°E), annual deforestation rates increased by ~32% between 2005 and 2020, rising from ~1.36 million ha to ~1.79 million ha per year due to logging, farming, and mining; if this trend continues, over 25% of the remaining rainforest could disappear by 2050.",
    "In the U.S. Great Plains (~36°N, 100°W), severe drought and poor farming practices during the 1930s Dust Bowl stripped topsoil from about 40 million hectares of farmland (around 100 million acres), rendering ~14 million hectares unusable by 1934.",
    "Urban sprawl led to conversion of green spaces into commercial zones.",
    "In the United States (~38°N, 97°W), about 12.5 million hectares of agricultural land (roughly 31 million acres) were irreversibly lost to development between 1992 and 2012 – an area nearly the size of Iowa – as urban sprawl expanded over farmland.",
    "In Costa Rica (~10°N, 84°W), forest cover rebounded from around 21% of the land in the 1980s to over 50% by 2013, regaining roughly 1.5 million hectares of forest thanks to aggressive conservation and reforestation policies that reversed decades of deforestation.",
    "Côte d’Ivoire (~7°N, 5°W) has lost more than 80% of its forest cover since 1960 (dropping from ~12 million to ~2 million hectares remaining), mainly due to cocoa farming, giving it one of the highest deforestation rates in Africa.",
    "In Borneo (~0°, 114°E), tropical forest cover declined by over 30% between the 1970s and mid-2010s – a loss of roughly 16 million hectares – as logging and oil palm plantations replaced vast areas of rainforest.",
    "Worldwide, at least 64% of wetlands have disappeared since 1900, largely due to drainage for agriculture and urban development, with losses exceeding 80% in some regions despite their ecological importance.",


    
    "In the Amazon Basin (~3°S, 60°W), between 2001 and 2020, forest cover decreased by ~9%, representing a loss of over 54.2 million hectares, largely due to deforestation."

]

print("Testing entity extraction on new sentences:\n")
for sentence in test_sentences:
    print(f"Text: {sentence}")
    entities = extract_entities(sentence)
    print("Extracted entities:")
    for ent in entities:
        print(f"  - '{ent['text']}' → {ent['label']}")
    print()

Testing entity extraction on new sentences:

Text: In the Congo Basin (~0°, 20°E), annual deforestation rates increased by ~32% between 2005 and 2020, rising from ~1.36 million ha to ~1.79 million ha per year due to logging, farming, and mining; if this trend continues, over 25% of the remaining rainforest could disappear by 2050.
Extracted entities:
  - 'Congo Basin' → LOC
  - 'annual' → DATE
  - 'deforestation' → PROCESS
  - 'increased' → CHANGE
  - '~32%' → PERCENT
  - 'between 2005 and 2020' → DATE
  - '1.36 million' → COORDINATES
  - 'ha' → SURFACE_UNIT
  - '1.79 million' → COORDINATES
  - 'ha' → SURFACE_UNIT
  - 'over 25%' → PERCENT
  - '2050' → DATE

Text: In the U.S. Great Plains (~36°N, 100°W), severe drought and poor farming practices during the 1930s Dust Bowl stripped topsoil from about 40 million hectares of farmland (around 100 million acres), rendering ~14 million hectares unusable by 1934.
Extracted entities:
  - 'U.S.' → LOC
  - '100' → CARDINAL
  - 'the 1930s' → DATE


In [3]:
!pip install pdfplumber==0.10.3 pdfminer.six==20231228

     |████████████████████████████████| 48 kB 1.8 MB/s eta 0:00:011
     |████████████████████████████████| 5.6 MB 3.9 MB/s eta 0:00:01
  Using cached pypdfium2-4.30.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.8 MB)
     |████████████████████████████████| 4.4 MB 4.1 MB/s eta 0:00:01
ERROR: pdfplumber 0.10.3 has requirement pdfminer.six==20221105, but you'll have pdfminer-six 20231228 which is incompatible.


In [5]:
!pip install PyMuPDF

In [30]:
import fitz  # PyMuPDF
import pandas as pd
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForTokenClassification

nltk.download('punkt', quiet=True)

# Set device first
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer
model_path = "./final_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# IMPORTANT: Move model to the device
model.to(device)
model.eval()

# Get id2label mapping from model config
id2label = model.config.id2label

# Verify everything is set up correctly
try:
    model
    tokenizer
    device
    id2label
    print("✅ All required variables are available")
    print(f"Model device: {next(model.parameters()).device}")
    print(f"Device variable: {device}")
except NameError as e:
    print(f"❌ Missing variable: {e}")

# Define the PDF file path
pdf_file_path = 'relations_table_20250709_165543 - Sheet5.pdf'  # Update this path

def extract_sentences_from_pdf(file_path):
    """Extract sentences from a PDF file using PyMuPDF."""
    sentences = []
    
    try:
        doc = fitz.open(file_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text()
            
            if text:
                # Split text into sentences
                page_sentences = nltk.sent_tokenize(text)
                sentences.extend([s.strip() for s in page_sentences if s.strip()])
        
        doc.close()
    except Exception as e:
        print(f"Error reading PDF: {e}")
    
    return sentences

def predict_entities(text, model, tokenizer, device, id2label):
    """Test the model with new text"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=512
    )
    
    # Move input tensors to the same device as model
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    offset_mapping = inputs["offset_mapping"][0].tolist()
    
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Move predictions back to CPU for processing
    predictions = predictions.cpu()
    input_ids_cpu = input_ids.cpu()
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu[0])
    predicted_labels = [id2label[pred.item()] for pred in predictions[0]]
    
    entities = []
    current_entity = None
    
    for idx, (token, label, (start, end)) in enumerate(zip(tokens, predicted_labels, offset_mapping)):
        # Skip special tokens
        if token in tokenizer.all_special_tokens:
            continue
            
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "text": text[start:end],
                "label": label[2:],
                "start": start,
                "end": end
            }
        elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
            current_entity["end"] = end
            current_entity["text"] = text[current_entity["start"]:current_entity["end"]]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    return entities

# Extract sentences from PDF
print(f"📄 Reading PDF from: {pdf_file_path}")
sentences = extract_sentences_from_pdf(pdf_file_path)
print(f"📚 Extracted {len(sentences)} sentences from PDF\n")

# Process each sentence and collect results
results = []
all_output_data = []

print("🧪 Processing sentences and extracting entities...\n")
print("="*80)

for idx, sentence in enumerate(sentences):
    if not sentence or sentence.strip() == "":
        continue
    
    # Predict entities - pass all required parameters
    entities = predict_entities(sentence, model, tokenizer, device, id2label)
    
    # Display results for this sentence
    print(f"\n📄 Sentence {idx + 1}:")
    print(f"📝 Text: {sentence}")
    
    if entities:
        print("🏷️  Detected Entities:")
        for entity in entities:
            print(f"    • {entity['text']} → {entity['label']} (positions: {entity['start']}-{entity['end']})")
            
            # Collect data for DataFrame
            all_output_data.append({
                "Sentence_ID": idx + 1,
                "Sentence": sentence,
                "Entity": entity['text'],
                "Entity_Type": entity['label'],
                "Start_Position": entity['start'],
                "End_Position": entity['end']
            })
    else:
        print("    ❌ No entities detected")
        all_output_data.append({
            "Sentence_ID": idx + 1,
            "Sentence": sentence,
            "Entity": None,
            "Entity_Type": None,
            "Start_Position": None,
            "End_Position": None
        })
    
    print("-"*80)
    
    # Store result
    results.append({
        "sentence_id": idx + 1,
        "sentence": sentence,
        "entities": entities
    })

# Create DataFrame with results
df_results = pd.DataFrame(all_output_data)

# Summary statistics
print("\n📈 SUMMARY STATISTICS:")
print(f"Total sentences processed: {len(results)}")
total_entities = sum(len(r['entities']) for r in results)
print(f"Total entities detected: {total_entities}")

# Count entities by type
entity_counts = {}
for result in results:
    for entity in result['entities']:
        entity_type = entity['label']
        entity_counts[entity_type] = entity_counts.get(entity_type, 0) + 1

print("\nEntity counts by type:")
for entity_type, count in sorted(entity_counts.items()):
    print(f"  • {entity_type}: {count}")

# Display DataFrame
print("\n📊 Results DataFrame:")
display(df_results)

# Optional: Save to CSV
df_results.to_csv('entity_extraction_results.csv', index=False)
print("\n💾 Results saved to entity_extraction_results.csv")

Using device: cuda
✅ All required variables are available
Model device: cuda:0
Device variable: cuda
📄 Reading PDF from: relations_table_20250709_165543 - Sheet5.pdf
📚 Extracted 2 sentences from PDF

🧪 Processing sentences and extracting entities...


📄 Sentence 1:
📝 Text: sentenceID
sentence
source
source_type
relationship
target
target_type
confidence
1 After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible desertification and large parts of the Sahel were designated as degraded land (e.g
2  The forests of West and Central Africa probably originally covered a combined area of about 3 million km 2
3  Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3% over the study site area, and from 47% to 64% of the arable lands in the study site due to the loss fallows
4 Agriculture in this region has been dominated over the past three decad

NameError: name 'page_num' is not defined

In [31]:
import pandas as pd
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForTokenClassification

nltk.download('punkt', quiet=True)

# Set device first
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer
model_path = "./final_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# IMPORTANT: Move model to the device
model.to(device)
model.eval()

# Get id2label mapping from model config
id2label = model.config.id2label

# Verify everything is set up correctly
print("✅ All required variables are available")
print(f"Model device: {next(model.parameters()).device}")
print(f"Device variable: {device}")

# Define the CSV file path
csv_file_path = 'lulc_relations_EVALUATION_csv  - Roberto .csv'  # Update this path

def load_sentences_from_csv(file_path, sentence_column='sentence'):
    """Load sentences from CSV file."""
    try:
        # Read CSV file
        df = pd.read_csv(file_path)
        print(f"📄 CSV loaded successfully!")
        print(f"📊 Shape: {df.shape}")
        print(f"📋 Columns: {df.columns.tolist()}")
        
        # Check if the sentence column exists
        if sentence_column not in df.columns:
            print(f"❌ Column '{sentence_column}' not found!")
            print(f"Available columns: {df.columns.tolist()}")
            
            # Try to find a likely sentence column
            possible_cols = ['sentence', 'text', 'content', 'description']
            for col in possible_cols:
                if col in df.columns:
                    print(f"🔄 Using column '{col}' instead")
                    sentence_column = col
                    break
            else:
                print("Please specify the correct column name")
                return [], df
        
        # Extract sentences, removing any NaN values
        sentences = df[sentence_column].dropna().astype(str).tolist()
        
        # Filter out very short sentences or obvious non-sentences
        filtered_sentences = []
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 10 and not sentence.lower().startswith(('sentenceid', 'sentence')):
                filtered_sentences.append(sentence)
        
        print(f"📚 Extracted {len(filtered_sentences)} valid sentences from CSV\n")
        return filtered_sentences, df
        
    except Exception as e:
        print(f"❌ Error reading CSV: {e}")
        return [], None

def predict_entities(text, model, tokenizer, device, id2label):
    """Test the model with new text"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=512
    )
    
    # Move input tensors to the same device as model
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    offset_mapping = inputs["offset_mapping"][0].tolist()
    
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Move predictions back to CPU for processing
    predictions = predictions.cpu()
    input_ids_cpu = input_ids.cpu()
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu[0])
    predicted_labels = [id2label[pred.item()] for pred in predictions[0]]
    
    entities = []
    current_entity = None
    
    for idx, (token, label, (start, end)) in enumerate(zip(tokens, predicted_labels, offset_mapping)):
        # Skip special tokens
        if token in tokenizer.all_special_tokens:
            continue
            
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "text": text[start:end],
                "label": label[2:],
                "start": start,
                "end": end
            }
        elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
            current_entity["end"] = end
            current_entity["text"] = text[current_entity["start"]:current_entity["end"]]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    return entities

# Load sentences from CSV
print(f"📄 Reading CSV from: {csv_file_path}")
sentences, original_df = load_sentences_from_csv(csv_file_path, sentence_column='sentence')

if not sentences:
    print("❌ No sentences found. Please check your CSV file and column name.")
    exit()

# Process each sentence and collect results
results = []
all_output_data = []

print("🧪 Processing sentences and extracting entities...\n")
print("="*80)

for idx, sentence in enumerate(sentences):
    if not sentence or sentence.strip() == "":
        continue
    
    # Predict entities
    entities = predict_entities(sentence, model, tokenizer, device, id2label)
    
    # Display results for this sentence
    print(f"\n📄 Sentence {idx + 1}:")
    print(f"📝 Text: {sentence}")
    
    if entities:
        print("🏷️  Detected Entities:")
        for entity in entities:
            print(f"    • {entity['text']} → {entity['label']} (positions: {entity['start']}-{entity['end']})")
            
            # Collect data for DataFrame
            all_output_data.append({
                "Sentence_ID": idx + 1,
                "Sentence": sentence,
                "Entity": entity['text'],
                "Entity_Type": entity['label'],
                "Start_Position": entity['start'],
                "End_Position": entity['end']
            })
    else:
        print("    ❌ No entities detected")
        all_output_data.append({
            "Sentence_ID": idx + 1,
            "Sentence": sentence,
            "Entity": None,
            "Entity_Type": None,
            "Start_Position": None,
            "End_Position": None
        })
    
    print("-"*80)
    
    # Store result
    results.append({
        "sentence_id": idx + 1,
        "sentence": sentence,
        "entities": entities
    })

# Create DataFrame with results
df_results = pd.DataFrame(all_output_data)

# Summary statistics
print("\n📈 SUMMARY STATISTICS:")
print(f"Total sentences processed: {len(results)}")
total_entities = sum(len(r['entities']) for r in results)
print(f"Total entities detected: {total_entities}")

# Count entities by type
entity_counts = {}
for result in results:
    for entity in result['entities']:
        entity_type = entity['label']
        entity_counts[entity_type] = entity_counts.get(entity_type, 0) + 1

print("\nEntity counts by type:")
for entity_type, count in sorted(entity_counts.items()):
    print(f"  • {entity_type}: {count}")

# Display DataFrame
print("\n📊 Results DataFrame:")
display(df_results)

# Save results
output_filename = 'entity_extraction_results.csv'
df_results.to_csv(output_filename, index=False)
print(f"\n💾 Results saved to {output_filename}")

# Optional: Also save a summary
summary_df = pd.DataFrame([
    {"Metric": "Total Sentences", "Value": len(results)},
    {"Metric": "Total Entities", "Value": total_entities},
    {"Metric": "Sentences with Entities", "Value": len([r for r in results if r['entities']])},
    {"Metric": "Sentences without Entities", "Value": len([r for r in results if not r['entities']])}
])

for entity_type, count in entity_counts.items():
    summary_df = pd.concat([summary_df, pd.DataFrame([{"Metric": f"Entities: {entity_type}", "Value": count}])], ignore_index=True)

summary_df.to_csv('extraction_summary.csv', index=False)
print(f"📈 Summary saved to extraction_summary.csv")

Using device: cuda
✅ All required variables are available
Model device: cuda:0
Device variable: cuda
📄 Reading CSV from: lulc_relations_EVALUATION_csv  - Roberto .csv
📄 CSV loaded successfully!
📊 Shape: (20, 2)
📋 Columns: ['sentenceID', 'sentence']
📚 Extracted 20 valid sentences from CSV

🧪 Processing sentences and extracting entities...


📄 Sentence 1:
📝 Text: After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was often considered as irreversible desertification and large parts of the Sahel were designated as degraded land (e.g
🏷️  Detected Entities:
    • the 1970s and 1980s → DATE (positions: 22-41)
    • loss → CHANGE (positions: 56-60)
    • desertification → PROCESS (positions: 124-139)
--------------------------------------------------------------------------------

📄 Sentence 2:
📝 Text: The forests of West and Central Africa probably originally covered a combined area of about 3 million km 2
🏷️  Detected Entities:
    • forests → LULC (positi

,Sentence_ID,Sentence,Entity,Entity_Type,Start_Position,End_Position
0,1,"After the droughts in the 1970s and 1980s, the...",the 1970s and 1980s,DATE,22,41
1,1,"After the droughts in the 1970s and 1980s, the...",loss,CHANGE,56,60
2,1,"After the droughts in the 1970s and 1980s, the...",desertification,PROCESS,124,139
3,2,The forests of West and Central Africa probabl...,forests,LULC,4,11
4,2,The forests of West and Central Africa probabl...,West,LOC,15,19
...,...,...,...,...,...,...
104,19,Ratios of fallow to cultivated area are 0.8 in...,fallow,LULC,218,224
105,19,Ratios of fallow to cultivated area are 0.8 in...,5,CARDINAL,244,245
106,19,Ratios of fallow to cultivated area are 0.8 in...,Sayero,LOC,249,255
107,20,Regular cultivation keeps the soil penetrable ...,cultivation,PROCESS,8,19



💾 Results saved to entity_extraction_results.csv
📈 Summary saved to extraction_summary.csv


In [75]:
def analyze_sentence_lengths(data, tokenizer):
    """Check how many sentences exceed the token limit"""
    print("🔍 ANALYZING SENTENCE LENGTHS...")
    
    short_sentences = 0
    medium_sentences = 0
    long_sentences = 0
    very_long_sentences = 0
    
    truncated_entities = 0
    total_entities = 0
    
    for item in data:
        sentence = item['original_sentence']
        entities = item['entities']
        
        # Tokenize to count tokens
        tokens = tokenizer.tokenize(sentence)
        token_count = len(tokens)
        
        total_entities += len(entities)
        
        # Count by length
        if token_count <= 128:
            short_sentences += 1
        elif token_count <= 256:
            medium_sentences += 1
        elif token_count <= 512:
            long_sentences += 1
        else:
            very_long_sentences += 1
            
            # Check for truncated entities
            encoding = tokenizer(sentence, truncation=True, max_length=512, return_offsets_mapping=True)
            last_char = encoding['offset_mapping'][-1][1] if encoding['offset_mapping'] else 0
            
            for entity in entities:
                if entity['start_char'] >= last_char:
                    truncated_entities += 1
    
    total_sentences = len(data)
    
    print(f"\n📊 SENTENCE LENGTH DISTRIBUTION:")
    print(f"  Short (≤128 tokens): {short_sentences} ({short_sentences/total_sentences*100:.1f}%)")
    print(f"  Medium (129-256 tokens): {medium_sentences} ({medium_sentences/total_sentences*100:.1f}%)")
    print(f"  Long (257-512 tokens): {long_sentences} ({long_sentences/total_sentences*100:.1f}%)")
    print(f"  Very Long (>512 tokens): {very_long_sentences} ({very_long_sentences/total_sentences*100:.1f}%)")
    
    print(f"\n⚠️ TRUNCATION IMPACT:")
    print(f"  Sentences that get truncated: {very_long_sentences}")
    print(f"  Entities lost due to truncation: {truncated_entities}")
    print(f"  Entity loss rate: {truncated_entities/total_entities*100:.1f}%")
    
    return {
        'very_long_count': very_long_sentences,
        'truncated_entities': truncated_entities,
        'total_entities': total_entities
    }

# Test your data
length_analysis = analyze_sentence_lengths(fixed_raw_data, tokenizer)

Token indices sequence length is longer than the specified maximum sequence length for this model (710 > 512). Running this sequence through the model will result in indexing errors


🔍 ANALYZING SENTENCE LENGTHS...

📊 SENTENCE LENGTH DISTRIBUTION:
  Short (≤128 tokens): 14229 (98.0%)
  Medium (129-256 tokens): 249 (1.7%)
  Long (257-512 tokens): 35 (0.2%)
  Very Long (>512 tokens): 9 (0.1%)

⚠️ TRUNCATION IMPACT:
  Sentences that get truncated: 9
  Entities lost due to truncation: 554
  Entity loss rate: 1.5%


In [16]:
# First, make sure tqdm is installed
!pip install tqdm -q

from tqdm import tqdm

def extract_all_lulc_terms(texts, model_path="./final_ner_model"):
    """
    Extract all unique LULC terms from a list of texts.
    This gives you the comprehensive list you requested.
    """
    all_lulc_terms = []
    
    # Add progress bar for processing texts
    print("🔄 Extracting entities from texts...")
    for text in tqdm(texts, desc="Processing sentences", unit="sentence"):
        entities = extract_entities(text, model_path)
        lulc_entities = [ent['text'].lower() for ent in entities if ent['label'] == 'LULC']
        all_lulc_terms.extend(lulc_entities)
    
    # Get unique terms
    unique_lulc_terms = sorted(list(set(all_lulc_terms)))
    return unique_lulc_terms

# Extract LULC terms from your original data
all_texts = [item['original_sentence'] for item in fixed_raw_data]
print(f"📊 Total sentences to process: {len(all_texts)}")

unique_lulc_terms = extract_all_lulc_terms(all_texts)

print(f"\n✅ Found {len(unique_lulc_terms)} unique LULC terms in your data:\n")

# Show first 20 terms
print("First 20 LULC terms:")
for i, term in enumerate(unique_lulc_terms[:20], 1):
    print(f"  {i}. {term}")

if len(unique_lulc_terms) > 20:
    print(f"  ... and {len(unique_lulc_terms) - 20} more terms")

# Save to file
with open('unique_lulc_terms.json', 'w') as f:
    json.dump(unique_lulc_terms, f, indent=2)
print(f"\n💾 Saved all {len(unique_lulc_terms)} terms to 'unique_lulc_terms.json'")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


📊 Total sentences to process: 14522
🔄 Extracting entities from texts...


Processing sentences: 100%|██████████████████████████████████████████████████████████████| 14522/14522 [56:45<00:00,  4.26sentence/s]


✅ Found 111 unique LULC terms in your data:

First 20 LULC terms:
  1. agriculture
  2. agroforestry
  3. annual crops
  4. bare soil
  5. bare soils
  6. barley
  7. beans
  8. build-up area
  9. built up area
  10. built up areas
  11. built-up area
  12. built-up areas
  13. burned area
  14. burnt area
  15. burnt areas
  16. c
  17. cabbage
  18. cashew
  19. cereal
  20. cities
  ... and 91 more terms

💾 Saved all 111 terms to 'unique_lulc_terms.json'


In [ ]:
def extract_entities(text, model_path="./final_ner_model"):
    """
    Extract entities from new text using the trained model.
    This function shows how to use your model in practice.
    """
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.eval()
    
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
        padding=True
    )
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**{k: v for k, v in inputs.items() if k != 'offset_mapping'})
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Convert predictions to labels
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    labels = [model.config.id2label[pred.item()] for pred in predictions[0]]
    offset_mapping = inputs['offset_mapping'][0]
    
    # Extract entities
    entities = []
    current_entity = None
    
    for idx, (token, label, offset) in enumerate(zip(tokens, labels, offset_mapping)):
        if token in tokenizer.all_special_tokens:
            continue
            
        if label.startswith('B-'):
            # Start of new entity
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'text': tokenizer.convert_tokens_to_string([token]),
                'label': label[2:],
                'start': offset[0].item(),
                'end': offset[1].item(),
                'tokens': [token]
            }
        elif label.startswith('I-') and current_entity and label[2:] == current_entity['label']:
            # Continuation of current entity
            current_entity['tokens'].append(token)
            current_entity['end'] = offset[1].item()
            current_entity['text'] = tokenizer.convert_tokens_to_string(current_entity['tokens'])
        else:
            # Not an entity
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    # Clean up entity text
    for entity in entities:
        entity['text'] = entity['text'].strip()
        del entity['tokens']
    
    return entities

In [ ]:
# This cell demonstrates why the context-aware model is better

test_text = """
The metropolitan area experienced rapid growth. Build up areas expanded by 40% 
while agricultural zones decreased. The green belt around the city was converted 
to residential use. This urban development affected local ecosystems and build up area.
"""

print("Comparison: Context-Aware Model vs Simple Pattern Matching\n")
print(f"Test text: {test_text}\n")

# Extract with your new model
entities = extract_entities(test_text)

print("Context-Aware Model Results:")
lulc_found = [ent for ent in entities if ent['label'] == 'LULC']
for ent in lulc_found:
    print(f"  ✓ Found LULCPROCESS: '{ent['text']}'")

print(f"\nTotal LULC entities found: {len(lulc_found)}")

# Compare with LULC terms from your training data
print("\n📊 Comparison with training data LULC terms:")

# Get LULC terms that were in your original training data
training_lulc_terms = []
for item in raw_data:
    for entity in item['entities']:
        if entity['label'] == 'LULC':
            training_lulc_terms.append(entity['text'].lower())

training_lulc_set = set(training_lulc_terms)
found_lulc_set = set([ent['text'].lower() for ent in lulc_found])

new_terms = found_lulc_set - training_lulc_set
if new_terms:
    print(f"\n🆕 NEW LULC terms found (not in training data): {list(new_terms)}")
    print("This shows the model can generalize beyond exact training examples!")

In [6]:
import pandas as pd
import json
from transformers import pipeline

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero'
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] != 0].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Load your fixed model for entity extraction
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model"
)

# Function to clean entity output
def clean_entity_output(text, entities):
    """
    Converts wordpiece-based entities back to original text tokens.
    Removes special characters like 'Ġ' and aligns spans with original text.
    """
    cleaned_entities = []
    
    for entity in entities:
        # Get the actual text span from the original text
        start_char = entity['start']
        end_char = entity['end']
        entity_text = text[start_char:end_char]
        
        # Remove special token markers if present
        clean_text = entity_text.replace('Ġ', ' ').strip()
        
        cleaned_entities.append({
            'text': clean_text,
            'label': entity['entity'],
            'start_char': start_char,
            'end_char': end_char
        })
    
    return cleaned_entities

# Process each text_segment in the filtered DataFrame
results = []

print("\n📊 Processing and extracting entities from filtered sentences...")

for index, row in filtered_df.iterrows():
    article_id = row['id_segment']
    original_sentence = row['text_segment']
    print(f"\nProcessing ID: {article_id}")
    print(f"Text: {original_sentence[:100]}...")  # Show first 100 chars
    
    # Extract entities
    entities = ner_pipeline(original_sentence)
    
    # Clean the entity output
    cleaned_entities = clean_entity_output(original_sentence, entities)
    
    # Prepare the output structure
    output_entry = {
        "article_id": article_id,
        "original_sentence": original_sentence,
        "entities": []  # List to hold cleaned entities
    }
    
    # Add ALL entities to the output (including 'O' labels)
    for ent in cleaned_entities:
        entity_dict = {
            "text": ent['text'],
            "label": ent['label'],
            "start_char": ent['start_char'],
            "end_char": ent['end_char']
        }
        output_entry["entities"].append(entity_dict)
    
    results.append(output_entry)
    
    # Print ALL entities for verification
    print("Extracted entities:")
    for ent in output_entry['entities']:
        print(f"  - text: '{ent['text']}'")
        print(f"    label: {ent['label']}")
        print(f"    start_char: {ent['start_char']}")
        print(f"    end_char: {ent['end_char']}")

# Save results as JSON
output_json_file = 'model_extraction_results.json'
with open(output_json_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Processing complete. Results saved to '{output_json_file}' as JSON.")

# Optional: Print summary of all unique labels found
all_labels = set()
for result in results:
    for entity in result['entities']:
        all_labels.add(entity['label'])

print(f"\n📊 Summary - Unique labels found: {sorted(all_labels)}")

📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 803 rows


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



📊 Processing and extracting entities from filtered sentences...

Processing ID: 1-s2.0-S0301479717300713-main_226b
Text: #text': '(Lambin et al., 2003)'}], '#text': 'With high confidence, rainfall variability is a driving...
Extracted entities:
  - text: '2003'
    label: B-DATE
    start_char: 25
    end_char: 29
  - text: 'driving'
    label: B-CHANGE
    start_char: 93
    end_char: 100

Processing ID: 1-s2.0-S0303243414001718-main_19b
Text: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Extracted entities:
  - text: 'the'
    label: B-DATE
    start_char: 31
    end_char: 34
  - text: '1970'
    label: I-DATE
    start_char: 35
    end_char: 39
  - text: 's'
    label: I-DATE
    start_char: 39
    end_char: 40
  - text: 'and'
    label: I-DATE
    start_char: 41
    end_char: 44
  - text: '1980'
    label: I-DATE
    start_char: 45
    end_char: 49
  - text: 's'
    label: I-DATE
    start_char: 49
    end_char: 50
  - text

In [ ]:
import pandas as pd
import json
from transformers import pipeline
import os
import xml.sax.saxutils

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero' (optional)
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] != 0].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Load your NER model
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model",
    aggregation_strategy="simple"  # Groups sub-word tokens
)

def create_label_studio_config(all_labels, output_dir):
    """Create Label Studio config XML file"""
    
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", 
        "#FF9FF3", "#A55EEA", "#54A0FF", "#5F27CD", "#00D2D3",
        "#FF9F43", "#10AC84", "#EE5A24", "#0098C7", "#8395A7"
    ]
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    config_path = os.path.join(output_dir, "ner_label_studio_config.xml")
    
    # Build entity labels
    entity_labels = ""
    for i, label in enumerate(sorted(all_labels)):
        color = colors[i % len(colors)]
        # Escape XML special characters
        safe_label = xml.sax.saxutils.escape(label)
        entity_labels += f'    <Label value="{safe_label}" background="{color}"/>\n'
    
    # Build complete XML config
    label_studio_config = f"""<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
{entity_labels}  </Labels>
</View>"""
    
    # Write the file
    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(label_studio_config)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    return config_path

# Process each text_segment
label_studio_data = []
all_unique_labels = set()
annotation_id_counter = 1

print("\n📊 Processing texts and extracting entities...")

for index, row in filtered_df.iterrows():
    text = row['text_segment']
    article_id = row['id_segment']
    
    print(f"\nProcessing ID: {article_id}")
    print(f"Text preview: {text[:100]}...")
    
    # Extract entities using your model
    entities = ner_pipeline(text)
    
    # Build result array for this text
    results = []
    entity_counter = 0
    
    for entity in entities:
        # Skip 'O' labels
        if entity.get('entity_group', entity.get('entity', '')).endswith('-O'):
            continue
        
        # Extract clean label (remove B-, I- prefixes)
        label = entity.get('entity_group', entity.get('entity', '')).replace('B-', '').replace('I-', '')
        
        # Add to unique labels set
        all_unique_labels.add(label)
        
        # Create entity result
        result = {
            "id": f"entity_{entity_counter}",
            "type": "labels",
            "value": {
                "start": entity['start'],
                "end": entity['end'],
                "text": text[entity['start']:entity['end']],
                "labels": [label]
            },
            "from_name": "label",
            "to_name": "text"
        }
        results.append(result)
        entity_counter += 1
    
    # Create the task in Label Studio format
    task = {
        "data": {
            "text": text,
            "article_id": article_id  # Keep original ID for reference
        }
    }
    
    # Add annotations only if entities were found
    if results:
        task["annotations"] = [{
            "id": annotation_id_counter,
            "result": results
        }]
        annotation_id_counter += 1
    
    label_studio_data.append(task)
    
    # Print extracted entities for verification
    if results:
        print(f"Found {len(results)} entities:")
        for result in results[:5]:  # Show first 5
            print(f"  - '{result['value']['text']}' → {result['value']['labels'][0]}")
        if len(results) > 5:
            print(f"  ... and {len(results) - 5} more")
    else:
        print("No entities found")

# Create output directory
output_dir = "label_studio_output"
os.makedirs(output_dir, exist_ok=True)

# Save as Label Studio JSON format
output_file = os.path.join(output_dir, 'label_studio_ner_tasks.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(label_studio_data, f, indent=2, ensure_ascii=False)

# Generate Label Studio config file
if all_unique_labels:
    config_file = create_label_studio_config(all_unique_labels, output_dir)
else:
    print("⚠️ No labels found to generate config")

print(f"\n✅ Processing complete!")
print(f"📄 Results saved to: {output_file}")
print(f"📊 Total tasks: {len(label_studio_data)}")

# Print summary statistics
total_entities = sum(len(task.get('annotations', [{}])[0].get('result', [])) 
                    for task in label_studio_data if 'annotations' in task)
print(f"🏷️ Total entities extracted: {total_entities}")
print(f"🏷️ Unique labels found: {sorted(all_unique_labels)}")

print("\n📝 To import in Label Studio:")
print("1. Create a new project in Label Studio")
print(f"2. Go to Settings > Labeling Interface > Code")
print(f"3. Paste the content from: {output_dir}/ner_label_studio_config.xml")
print(f"4. Go to Import and upload: {output_file}")

# Show sample of output format
print("\n📋 Sample output format:")
if label_studio_data:
    print(json.dumps(label_studio_data[0], indent=2)[:500] + "...")

📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 803 rows


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



📊 Processing texts and extracting entities...

Processing ID: 1-s2.0-S0301479717300713-main_226b
Text preview: #text': '(Lambin et al., 2003)'}], '#text': 'With high confidence, rainfall variability is a driving...
Found 2 entities:
  - '2003' → DATE
  - 'driving' → CHANGE

Processing ID: 1-s2.0-S0303243414001718-main_19b
Text preview: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Found 2 entities:
  - 'the 1970s and 1980s' → DATE
  - 'loss' → CHANGE

Processing ID: 1-s2.0-S030438781000043X-mainext_313b
Text preview: #text': 'Few investment opportunities are available to most Rwandan farmers...
Found 1 entities:
  - 'Rwandan' → LOC

Processing ID: 1-s2.0-S095937809800003X-main_75b
Text preview: #text': 'Pastoral production has often existed in some sort of symbiotic interaction with the [agric...
Found 1 entities:
  - 'agriculture' → LULC

Processing ID: 1-s2.0-S0006320709005400-main_16b
Text preview: #text': 'The forests of We

In [4]:
import pandas as pd
import json
from transformers import pipeline, AutoTokenizer

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero'
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] != 'zero'].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Load your fixed model for entity extraction
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model",
    aggregation_strategy="none"  # This ensures we get token-level predictions
)

# Also load the tokenizer separately to get all tokens
tokenizer = AutoTokenizer.from_pretrained("./final_ner_model")

# Process each text_segment in the filtered DataFrame
results = []

print("\n📊 Processing and extracting entities from filtered sentences...")

for index, row in filtered_df.iterrows():
    article_id = row['id_segment']
    original_sentence = row['text_segment']
    print(f"\nProcessing ID: {article_id}")
    print(f"Text: {original_sentence[:100]}...")  # Show first 100 chars
    
    # Get NER predictions
    ner_results = ner_pipeline(original_sentence)
    
    # Tokenize the text to get all tokens
    tokens = tokenizer(original_sentence, return_offsets_mapping=True)
    
    # Create a list to hold all tokens with their labels
    all_tokens_with_labels = []
    
    # First, mark all tokens as 'O' by default
    for i, (start, end) in enumerate(tokens['offset_mapping']):
        if start == end:  # Skip special tokens
            continue
        token_text = original_sentence[start:end]
        all_tokens_with_labels.append({
            'text': token_text,
            'label': 'O',
            'start_char': start,
            'end_char': end
        })
    
    # Then update with actual entity labels
    for entity in ner_results:
        # Find which token this entity corresponds to
        for token_info in all_tokens_with_labels:
            if (token_info['start_char'] == entity['start'] and 
                token_info['end_char'] == entity['end']):
                token_info['label'] = entity['entity']
                break
    
    # Prepare the output structure
    output_entry = {
        "article_id": article_id,
        "original_sentence": original_sentence,
        "entities": all_tokens_with_labels
    }
    
    results.append(output_entry)
    
    # Print ALL tokens with their labels
    print("All tokens with labels:")
    for token in all_tokens_with_labels:
        print(f"  - text: '{token['text']}'")
        print(f"    label: {token['label']}")
        print(f"    start_char: {token['start_char']}")
        print(f"    end_char: {token['end_char']}")

# Save results as JSON
output_json_file = 'model_extraction_results_with_all_tokens.json'
with open(output_json_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Processing complete. Results saved to '{output_json_file}' as JSON.")

# Print summary of all unique labels found
all_labels = set()
for result in results:
    for entity in result['entities']:
        all_labels.add(entity['label'])

print(f"\n📊 Summary - Unique labels found: {sorted(all_labels)}")

📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 803 rows


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



📊 Processing and extracting entities from filtered sentences...

Processing ID: 1-s2.0-S0301479717300713-main_226b
Text: #text': '(Lambin et al., 2003)'}], '#text': 'With high confidence, rainfall variability is a driving...
All tokens with labels:
  - text: '#'
    label: O
    start_char: 0
    end_char: 1
  - text: 'text'
    label: O
    start_char: 1
    end_char: 5
  - text: '':'
    label: O
    start_char: 5
    end_char: 7
  - text: ''('
    label: O
    start_char: 8
    end_char: 10
  - text: 'L'
    label: O
    start_char: 10
    end_char: 11
  - text: 'amb'
    label: O
    start_char: 11
    end_char: 14
  - text: 'in'
    label: O
    start_char: 14
    end_char: 16
  - text: 'et'
    label: O
    start_char: 17
    end_char: 19
  - text: 'al'
    label: O
    start_char: 20
    end_char: 22
  - text: '.,'
    label: O
    start_char: 22
    end_char: 24
  - text: '2003'
    label: B-DATE
    start_char: 25
    end_char: 29
  - text: ')''
    label: O
    start_char: 2

Token indices sequence length is longer than the specified maximum sequence length for this model (712 > 512). Running this sequence through the model will result in indexing errors


All tokens with labels:
  - text: 'For'
    label: O
    start_char: 0
    end_char: 3
  - text: 'example'
    label: O
    start_char: 4
    end_char: 11
  - text: ','
    label: O
    start_char: 11
    end_char: 12
  - text: 'Pok'
    label: O
    start_char: 13
    end_char: 16
  - text: 'u'
    label: O
    start_char: 16
    end_char: 17
  - text: '-'
    label: O
    start_char: 17
    end_char: 18
  - text: 'Bo'
    label: O
    start_char: 18
    end_char: 20
  - text: 'ans'
    label: O
    start_char: 20
    end_char: 23
  - text: 'i'
    label: O
    start_char: 23
    end_char: 24
  - text: '&'
    label: O
    start_char: 25
    end_char: 26
  - text: 'Am'
    label: O
    start_char: 27
    end_char: 29
  - text: 'oak'
    label: O
    start_char: 29
    end_char: 32
  - text: 'o'
    label: O
    start_char: 32
    end_char: 33
  - text: '('
    label: O
    start_char: 34
    end_char: 35
  - text: '2015'
    label: B-DATE
    start_char: 35
    end_char: 39
  - text: 

In [14]:
# Save all important outputs

import os
os.makedirs("ner_output", exist_ok=True)

# 1. Model performance metrics
with open('ner_output/model_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

# 2. Unique LULC terms
with open('ner_output/unique_LULC_terms.txt', 'w') as f:
    for term in unique_lulc_terms:
        f.write(f"{term}\n")

# 3. Model configuration
model_info = {
    "base_model": "roberta-base",
    "num_labels": len(label2id),
    "labels": list(label2id.keys()),
    "training_sentences": len(dataset_split['train']),
    "validation_sentences": len(dataset_split['test']),
    "performance": {
        "f1_score": eval_results['eval_f1'],
        "precision": eval_results['eval_precision'],
        "recall": eval_results['eval_recall']
    }
}

with open('ner_output/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("✓ All outputs saved to 'ner_output/' directory:")
print("  - model_metrics.json: Detailed performance metrics")
print("  - unique_lulc_terms.txt: List of all unique LULC terms found")
print("  - model_info.json: Model configuration and summary")
print(f"\n✓ Trained model saved to './final_ner_model/'")
print("\nYou can now use extract_entities() function to process any new text!")

NameError: name 'eval_results' is not defined

In [ ]:
# Read the .txt file (assuming one term per line)
with open('ner_output/unique_PROCESS_terms.txt', 'r') as f:
    file1 = set(line.strip() for line in f)  # Correctly read and strip lines from .txt

# Read the .csv file
file2 = pd.read_csv('LCprocess.csv', header=None, names=['term'])
set2 = set(file2['term'])  # Convert DataFrame column to a set

# Find unique terms in file1 (txt) not in file2 (csv)
unique_to_file1 = file1 - set2

# Find unique terms in file2 (csv) not in file1 (txt)
unique_to_file2 = set2 - file1

# Print the results
print("Unique terms in file1.txt not in file2.csv:")
print(unique_to_file1)

print("\nUnique terms in file2.csv not in file1.txt:")
print(unique_to_file2)

In [ ]:
# Read the .txt file (assuming one term per line)
with open('ner_output/unique_lulc_terms.txt', 'r') as f:
    file1 = set(line.strip() for line in f)  # Correctly read and strip lines from .txt

# Read the .csv file
file2 = pd.read_csv('LULC.csv', header=None, names=['term'])
set2 = set(file2['term'])  # Convert DataFrame column to a set

# Find unique terms in file1 (txt) not in file2 (csv)
unique_to_file1 = file1 - set2

# Find unique terms in file2 (csv) not in file1 (txt)
unique_to_file2 = set2 - file1

# Print the results
print("Unique terms in file1.txt not in file2.csv:")
print(unique_to_file1)

print("\nUnique terms in file2.csv not in file1.txt:")
print(unique_to_file2)

In [11]:
import pandas as pd

# Load CSV file
df = pd.read_csv('extracted_lulc_sentences_from_articles.csv')

# Display structure
print(f"CSV has {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows:")
print(df.head())

# Get first 10 sentences
first_10_sentences = df['lulc_sentence'].head(10).tolist()

# Test each sentence
print("\n🔍 Testing model on first 10 sentences:\n")
for i, sentence in enumerate(first_10_sentences, 1):
    article_id = df.iloc[i-1]['article_id']
    
    print(f"{i}. Article ID: {article_id}")
    print(f"   Sentence: {sentence}")
    
    # Extract entities
    entities = extract_entities(sentence, "./final_ner_model")
    
    if entities:
        print("   Found entities:")
        for ent in entities:
            print(f"   - '{ent['text']}' → {ent['label']}")
    else:
        print("   No entities found")
    print()

CSV has 2003 rows
Columns: ['article_id', 'lulc_sentence']

First few rows:
  article_id                                      lulc_sentence
0  Article_1  Simulation results reveal that the landscape o...
1  Article_1  The study observed a significant increase (12....
2  Article_1  On the contrary, forest cover declined drastic...
3  Article_1  Rapid population growth triggered by rural urb...
4  Article_1  Under the business as usual scenario, predicti...

🔍 Testing model on first 10 sentences:

1. Article ID: Article_1
   Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
   Found entities:
   - 'Thimphu' → LOC
   - 'city' → LULC
   - 'changed' → CHANGE
   - 'change' → CHANGE
   - '2050' → DATE

2. Article ID: Article_1
   Sentence: The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight incre

In [33]:
# Quick diagnostic: Check if you're using the right model
def diagnose_model_issue(model_path="./final_ner_model"):
    from transformers import AutoConfig, AutoTokenizer, AutoModelForTokenClassification
    import torch
    
    # 1. Check model configuration
    config = AutoConfig.from_pretrained(model_path)
    print("🔍 MODEL DIAGNOSTIC:")
    print(f"Number of labels: {config.num_labels}")
    print(f"Label list: {config.id2label}")
    
    # Check if CARDINAL is in labels
    if 'CARDINAL' in config.id2label.values():
        print("\n❌ PROBLEM: 'CARDINAL' found in labels!")
        print("This is NOT your custom NER model - you're loading the wrong model!")
        return False
    
    # 2. Test specific tokens
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    
    # Test just "villages"
    test_text = "villages"
    inputs = tokenizer(test_text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]
    
    label = config.id2label[predictions[1].item()]  # Token 1 should be "villages"
    print(f"\nTest prediction for 'villages': {label}")
    
    if label != "B-LULC":
        print("❌ Model doesn't recognize 'villages' as LULC!")
        print("This confirms training data has missing labels.")
    
    return True

# Run diagnostic
is_correct_model = diagnose_model_issue()

🔍 MODEL DIAGNOSTIC:
Number of labels: 21
Label list: {0: 'O', 1: 'B-CHANGE', 2: 'I-CHANGE', 3: 'B-LOC', 4: 'I-LOC', 5: 'B-LULC', 6: 'I-LULC', 7: 'B-DATE', 8: 'I-DATE', 9: 'B-PERCENT', 10: 'I-PERCENT', 11: 'B-CARDINAL', 12: 'I-CARDINAL', 13: 'B-COORDINATES', 14: 'I-COORDINATES', 15: 'B-SURFACE_UNIT', 16: 'I-SURFACE_UNIT', 17: 'B-PROCESS', 18: 'I-PROCESS', 19: 'B-QUANTITY', 20: 'I-QUANTITY'}

Test prediction for 'villages': O
❌ Model doesn't recognize 'villages' as LULC!
This confirms training data has missing labels.


In [20]:
import json
import re
from collections import Counter

# Load your training data
with open('extracted_ALL_entities_structured.json', 'r') as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training sentences")

# Define LULC patterns that MUST be labeled
MUST_BE_LULC = [
    # Basic land types
    r'\b(agricultural|arable)\s+(fields?|lands?|areas?|zones?)\b',
    r'\b(urban|rural|suburban|industrial)\s+(areas?|zones?|lands?|districts?)\b',
    r'\b(built-up|built up|build up)\s+(areas?|lands?|zones?)\b',
    
    
    # Specific terms
    r'\bvillages?\b',
    r'\bsettlements?\b',
    r'\btowns?\b',
    r'\bcities\b',
    r'\bfarmlands?\b',
    r'\bcroplands?\b',
    r'\bforests?\b',
    r'\bwoodlands?\b',
    r'\bgrasslands?\b',
    r'\bwetlands?\b',
    r'\bmarshlands?\b',
    r'\bswamps?\b',
    r'\bpastures?\b',
    r'\borchards?\b',
    r'\bplantations?\b',
    r'\b(paddy|rice)\s+fields?\b',
    r'\b(green|open)\s+(spaces?|areas?)\b',
    r'\bparks?\b',
    r'\bgardens?\b',
    r'\bhabitats?\b'
     r'\burned area
acacias
bare rock
shrub form
shrublands
evergreen trees
]

# Find and fix missing LULC labels
fixed_count = 0
missing_examples = []

for item in training_data:
    sentence = item['original_sentence']
    
    # Get existing LULC entities
    existing_lulc = {(e['start_char'], e['end_char']): e['text'] 
                     for e in item['entities'] if e['label'] == 'LULC'}
    
    # Check each pattern
    for pattern in MUST_BE_LULC:
        for match in re.finditer(pattern, sentence, re.IGNORECASE):
            start, end = match.span()
            matched_text = match.group()
            
            # Check if this span is already labeled
            is_labeled = any(s <= start and end <= e for s, e in existing_lulc.keys())
            
            if not is_labeled:
                # Add missing LULC entity
                item['entities'].append({
                    'text': matched_text,
                    'label': 'LULC',
                    'start_char': start,
                    'end_char': end
                })
                fixed_count += 1
                
                if len(missing_examples) < 10:
                    missing_examples.append(f"'{matched_text}' in: {sentence[:80]}...")
    
    # Sort entities by position
    item['entities'] = sorted(item['entities'], key=lambda x: x['start_char'])

print(f"\n✅ Fixed {fixed_count} missing LULC labels!")
print("\nExamples of fixed labels:")
for ex in missing_examples:
    print(f"  - {ex}")

Loaded 13660 training sentences

✅ Fixed 1596 missing LULC labels!

Examples of fixed labels:
  - 'built-up area' in: The study observed a significant increase (12.77%) in built-up area from 2002 (5...
  - 'built up area' in: Under the business as usual scenario, prediction analysis for the year 2050 show...
  - 'agricultural fields' in: Prior to this, the area was occupied by terraced agricultural fields and some 13...
  - 'villages' in: Prior to this, the area was occupied by terraced agricultural fields and some 13...
  - 'plantation' in: In addition, records of major incidents such as disasters (fire, floods), planta...
  - 'agricultural lands' in: Assuming that these probabilities | Results and discussion: | Classification acc...
  - 'built-up area' in: Using the outputs from remote sensing imagery, field surveys, and topped expert ...
  - 'rural areas' in: The resultant improvement in basic facilities combined with creation of addition...
  - 'settlements' in: The resultant impro

In [21]:
# Check if our problematic sentence is now fixed
test_sentence = "Prior to this, the area was occupied by terraced agricultural fields and some 13 villages."

# Find this sentence in training data
found = False
for item in training_data:
    if test_sentence in item['original_sentence']:
        print(f"\nFound sentence in training data:")
        print(f"Sentence: {item['original_sentence']}")
        print(f"LULC entities:")
        for e in item['entities']:
            if e['label'] == 'LULC':
                print(f"  - '{e['text']}'")
        found = True
        break

if not found:
    print("\n⚠️ This exact sentence not in training data, but similar patterns are now fixed")

# Save corrected training data
with open('training_data_corrected.json', 'w') as f:
    json.dump(training_data, f, indent=2)
print("\n💾 Saved corrected training data to 'training_data_corrected.json'")


Found sentence in training data:
Sentence: Prior to this, the area was occupied by terraced agricultural fields and some 13 villages.
LULC entities:
  - 'agricultural fields'
  - 'villages'

💾 Saved corrected training data to 'training_data_corrected.json'


In [22]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

# Load tokenizer and initialize new model
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Your label mappings (make sure these match!)
label_list = ['O', 'B-CHANGE', 'I-CHANGE', 'B-LOC', 'I-LOC', 'B-LULC', 'I-LULC', 
              'B-DATE', 'I-DATE', 'B-PERCENT', 'I-PERCENT', 'B-CARDINAL', 'I-CARDINAL',
              'B-COORDINATES', 'I-COORDINATES', 'B-SURFACE_UNIT', 'I-SURFACE_UNIT',
              'B-PROCESS', 'I-PROCESS', 'B-QUANTITY', 'I-QUANTITY']

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Initialize model
model = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Prepare dataset with corrected data
def prepare_dataset(data, tokenizer, label2id):
    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(
            examples["tokens"],
            truncation=True,
            is_split_into_words=True,
            max_length=512,
            padding=True
        )
        
        labels = []
        for i, label in enumerate(examples["labels"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            label_ids = []
            previous_word_idx = None
            
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[label[word_idx]])
                else:
                    # For subword tokens, use the same label or -100
                    label_ids.append(label2id[label[word_idx]])
                previous_word_idx = word_idx
            
            labels.append(label_ids)
        
        tokenized_inputs["labels"] = labels
        return tokenized_inputs
    
    # Convert your data format
    processed_examples = []
    for item in data:
        # Simple tokenization for alignment
        tokens = item['original_sentence'].split()
        labels = ['O'] * len(tokens)
        
        # Apply entity labels
        for entity in item['entities']:
            # Find which tokens this entity spans
            # (This is simplified - you may need more sophisticated alignment)
            entity_text = entity['text']
            entity_label = entity['label']
            
            # Add proper B- and I- prefixes
            # ... (implement proper token-label alignment)
        
        processed_examples.append({
            'tokens': tokens,
            'labels': labels
        })
    
    # Create dataset
    dataset = Dataset.from_list(processed_examples)
    tokenized_dataset = dataset.map(
        lambda x: tokenize_and_align_labels(x),
        batched=True
    )
    
    return tokenized_dataset

# Prepare corrected dataset
print("Preparing dataset with corrected labels...")
dataset = prepare_dataset_enhanced(raw_data, tokenizer, label2id)
train_test = dataset.train_test_split(test_size=0.1)

# Training arguments
training_args = TrainingArguments(
    output_dir="./roberta-ner-corrected",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test["train"],
    eval_dataset=train_test["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

# Train!
print("\n🚀 Starting training with corrected labels...")
trainer.train()

# Save the fixed model
trainer.save_model("./final_ner_model_corrected")
print("\n✅ Model retrained and saved to './final_ner_model_corrected'")

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Preparing dataset with corrected labels...


Map:   0%|          | 0/13660 [00:00<?, ? examples/s]

AssertionError: You need to instantiate RobertaTokenizerFast with add_prefix_space=True to use it with pretokenized inputs.

In [11]:
import pandas as pd
import json
from transformers import pipeline
import os
import xml.sax.saxutils

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero' (optional)
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] != 0].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Load your NER model
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model",
    aggregation_strategy="simple"  # Groups sub-word tokens
)

def create_label_studio_config(all_labels, output_dir):
    """Create Label Studio config XML file"""
    
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", 
        "#FF9FF3", "#A55EEA", "#54A0FF", "#5F27CD", "#00D2D3",
        "#FF9F43", "#10AC84", "#EE5A24", "#0098C7", "#8395A7"
    ]
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    config_path = os.path.join(output_dir, "ner_label_studio_config.xml")
    
    # Build entity labels
    entity_labels = ""
    for i, label in enumerate(sorted(all_labels)):
        color = colors[i % len(colors)]
        # Escape XML special characters
        safe_label = xml.sax.saxutils.escape(label)
        entity_labels += f'    <Label value="{safe_label}" background="{color}"/>\n'
    
    # Build complete XML config
    label_studio_config = f"""<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
{entity_labels}  </Labels>
</View>"""
    
    # Write the file
    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(label_studio_config)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    return config_path

# Process each text_segment
label_studio_data = []
all_unique_labels = set()
annotation_id_counter = 1

print("\n📊 Processing texts and extracting entities...")

for index, row in filtered_df.iterrows():
    text = row['text_segment']
    article_id = row['id_segment']
    
    print(f"\nProcessing ID: {article_id}")
    print(f"Text preview: {text[:100]}...")
    
    # Extract entities using your model
    entities = ner_pipeline(text)
    
    # Build result array for this text
    results = []
    entity_counter = 0
    
    for entity in entities:
        # Skip 'O' labels
        if entity.get('entity_group', entity.get('entity', '')).endswith('-O'):
            continue
        
        # Extract clean label (remove B-, I- prefixes)
        label = entity.get('entity_group', entity.get('entity', '')).replace('B-', '').replace('I-', '')
        
        # Add to unique labels set
        all_unique_labels.add(label)
        
        # Create entity result
        result = {
            "id": f"entity_{entity_counter}",
            "type": "labels",
            "value": {
                "start": entity['start'],
                "end": entity['end'],
                "text": text[entity['start']:entity['end']],
                "labels": [label]
            },
            "from_name": "label",
            "to_name": "text"
        }
        results.append(result)
        entity_counter += 1
    
    # Create the task in Label Studio format
    task = {
        "data": {
            "text": text,
            "article_id": article_id  # Keep original ID for reference
        }
    }
    
    # Add annotations only if entities were found
    if results:
        task["annotations"] = [{
            "id": annotation_id_counter,
            "result": results
        }]
        annotation_id_counter += 1
    
    label_studio_data.append(task)
    
    # Print extracted entities for verification
    if results:
        print(f"Found {len(results)} entities:")
        for result in results[:5]:  # Show first 5
            print(f"  - '{result['value']['text']}' → {result['value']['labels'][0]}")
        if len(results) > 5:
            print(f"  ... and {len(results) - 5} more")
    else:
        print("No entities found")

# Create output directory
output_dir = "label_studio_output"
os.makedirs(output_dir, exist_ok=True)

# Save as Label Studio JSON format
output_file = os.path.join(output_dir, 'label_studio_ner_tasks.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(label_studio_data, f, indent=2, ensure_ascii=False)

# Generate Label Studio config file
if all_unique_labels:
    config_file = create_label_studio_config(all_unique_labels, output_dir)
else:
    print("⚠️ No labels found to generate config")

print(f"\n✅ Processing complete!")
print(f"📄 Results saved to: {output_file}")
print(f"📊 Total tasks: {len(label_studio_data)}")

# Print summary statistics
total_entities = sum(len(task.get('annotations', [{}])[0].get('result', [])) 
                    for task in label_studio_data if 'annotations' in task)
print(f"🏷️ Total entities extracted: {total_entities}")
print(f"🏷️ Unique labels found: {sorted(all_unique_labels)}")

print("\n📝 To import in Label Studio:")
print("1. Create a new project in Label Studio")
print(f"2. Go to Settings > Labeling Interface > Code")
print(f"3. Paste the content from: {output_dir}/ner_label_studio_config.xml")
print(f"4. Go to Import and upload: {output_file}")

# Show sample of output format
print("\n📋 Sample output format:")
if label_studio_data:
    print(json.dumps(label_studio_data[0], indent=2)[:500] + "...")

📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 233 rows


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



📊 Processing texts and extracting entities...

Processing ID: 1-s2.0-S0303243414001718-main_19b
Text preview: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Found 2 entities:
  - 'the 1970s and 1980s' → DATE
  - 'loss' → CHANGE

Processing ID: 1-s2.0-S095937809800003X-main_75b
Text preview: #text': 'Pastoral production has often existed in some sort of symbiotic interaction with the [agric...
Found 1 entities:
  - 'agriculture' → LULC

Processing ID: 1-s2.0-S0006320709005400-main_16b
Text preview: #text': 'The forests of West and Central Africa probably originally covered a combined area of about...
Found 5 entities:
  - 'forests' → LULC
  - 'West' → LOC
  - 'Central Africa' → LOC
  - 'area of about 3 million km' → SURFACE_UNIT
  - '2' → CARDINAL

Processing ID: 1-s2.0-S095937809800003X-main_8
Text preview: (1993)'}], '#text': "An increase of land under [cultivation] can be in the form of either (a) cleari...
Found 4 entities:
 

In [12]:
import pandas as pd
import json
from transformers import pipeline
import os
import xml.sax.saxutils

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero' (optional)
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] == 2].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Load your NER model
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model",
    aggregation_strategy="simple"  # Groups sub-word tokens
)

def create_label_studio_config(all_labels, output_dir):
    """Create Label Studio config XML file"""
    
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", 
        "#FF9FF3", "#A55EEA", "#54A0FF", "#5F27CD", "#00D2D3",
        "#FF9F43", "#10AC84", "#EE5A24", "#0098C7", "#8395A7"
    ]
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    config_path = os.path.join(output_dir, "ner_label_studio_config.xml")
    
    # Build entity labels
    entity_labels = ""
    for i, label in enumerate(sorted(all_labels)):
        color = colors[i % len(colors)]
        # Escape XML special characters
        safe_label = xml.sax.saxutils.escape(label)
        entity_labels += f'    <Label value="{safe_label}" background="{color}"/>\n'
    
    # Build complete XML config
    label_studio_config = f"""<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
{entity_labels}  </Labels>
</View>"""
    
    # Write the file
    with open(config_path, 'w', encoding='utf-8') as f:
        f.write(label_studio_config)
    
    print(f"✅ Label Studio config saved to: {config_path}")
    return config_path

# Process each text_segment
label_studio_data = []
all_unique_labels = set()
annotation_id_counter = 1

print("\n📊 Processing texts and extracting entities...")

for index, row in filtered_df.iterrows():
    text = row['text_segment']
    article_id = row['id_segment']
    
    print(f"\nProcessing ID: {article_id}")
    print(f"Text preview: {text[:100]}...")
    
    # Extract entities using your model
    entities = ner_pipeline(text)
    
    # Build result array for this text
    results = []
    entity_counter = 0
    
    for entity in entities:
        # Skip 'O' labels
        if entity.get('entity_group', entity.get('entity', '')).endswith('-O'):
            continue
        
        # Extract clean label (remove B-, I- prefixes)
        label = entity.get('entity_group', entity.get('entity', '')).replace('B-', '').replace('I-', '')
        
        # Add to unique labels set
        all_unique_labels.add(label)
        
        # Create entity result (simplified without from_name and to_name)
        result = {
            "id": f"entity_{entity_counter}",
            "type": "labels",
            "value": {
                "start": entity['start'],
                "end": entity['end'],
                "text": text[entity['start']:entity['end']],
                "labels": [label]
            }
        }
        results.append(result)
        entity_counter += 1
    
    # Create the task in Label Studio format
    task = {
        "data": {
            "text": text,
            "article_id": article_id  # Keep original ID for reference
        }
    }
    
    # Add annotations only if entities were found
    if results:
        task["annotations"] = [{
            "id": annotation_id_counter,
            "result": results
        }]
        annotation_id_counter += 1
    
    label_studio_data.append(task)
    
    # Print extracted entities for verification
    if results:
        print(f"Found {len(results)} entities:")
        for result in results[:5]:  # Show first 5
            print(f"  - '{result['value']['text']}' → {result['value']['labels'][0]}")
        if len(results) > 5:
            print(f"  ... and {len(results) - 5} more")
    else:
        print("No entities found")

# Create output directory
output_dir = "label_studio_output"
os.makedirs(output_dir, exist_ok=True)

# Save as Label Studio JSON format
output_file = os.path.join(output_dir, 'label_studio_ner_tasks.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(label_studio_data, f, indent=2, ensure_ascii=False)

# Generate Label Studio config file
if all_unique_labels:
    config_file = create_label_studio_config(all_unique_labels, output_dir)
else:
    print("⚠️ No labels found to generate config")

print(f"\n✅ Processing complete!")
print(f"📄 Results saved to: {output_file}")
print(f"📊 Total tasks: {len(label_studio_data)}")

# Print summary statistics
total_entities = sum(len(task.get('annotations', [{}])[0].get('result', [])) 
                    for task in label_studio_data if 'annotations' in task)
print(f"🏷️ Total entities extracted: {total_entities}")
print(f"🏷️ Unique labels found: {sorted(all_unique_labels)}")

print("\n📝 To import in Label Studio:")
print("1. Create a new project in Label Studio")
print(f"2. Go to Settings > Labeling Interface > Code")
print(f"3. Paste the content from: {output_dir}/ner_label_studio_config.xml")
print(f"4. Go to Import and upload: {output_file}")

# Show sample of output format
print("\n📋 Sample output format:")
if label_studio_data:
    print(json.dumps(label_studio_data[0], indent=2)[:500] + "...")

📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 67 rows


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.



📊 Processing texts and extracting entities...

Processing ID: 1-s2.0-S0303243414001718-main_19b
Text preview: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Found 2 entities:
  - 'the 1970s and 1980s' → DATE
  - 'loss' → CHANGE

Processing ID: 1-s2.0-S0006320709005400-main_16b
Text preview: #text': 'The forests of West and Central Africa probably originally covered a combined area of about...
Found 5 entities:
  - 'forests' → LULC
  - 'West' → LOC
  - 'Central Africa' → LOC
  - 'area of about 3 million km' → SURFACE_UNIT
  - '2' → CARDINAL

Processing ID: 1-s2.0-S0140196322000143-main_121
Text preview: ).'}]}, {'@xmlns': 'http://www.tei-c.org/ns/1.0', 'head': {'@n': '4.2.', '#text': '[Land use ]change...
Found 9 entities:
  - '4.2' → CARDINAL
  - 'change' → CHANGE
  - '1996 to 2017' → DATE
  - 'increased' → CHANGE
  - '40% to 51.3%' → PERCENT
  ... and 4 more

Processing ID: 1-s2.0-S0264837715302131-main_119
Text preview: [Agric

In [15]:
import spacy
import pandas as pd
import json
from transformers import pipeline
from itertools import groupby

# Load your NER model
ner_pipeline = pipeline(
    "ner",
    model="./final_ner_model",  # Update with your model path
    tokenizer="./final_ner_model",
    aggregation_strategy="simple"  # Groups sub-word tokens
)

def ner_to_spans(text, entities):
    """Convert NER output to Label Studio spans format"""
    results = []
    unique_labels = set()
    
    for entity in entities:
        # Skip 'O' labels
        if entity.get('entity_group', entity.get('entity', '')).endswith('-O'):
            continue
            
        # Extract clean label (remove B-, I- prefixes)
        label = entity.get('entity_group', entity.get('entity', '')).replace('B-', '').replace('I-', '')
        
        # Add to unique labels set
        unique_labels.add(label)
        
        # Create span in Label Studio format
        results.append({
            'from_name': 'label',
            'to_name': 'text',
            'type': 'labels',
            'value': {
                'start': entity['start'],
                'end': entity['end'],
                'text': text[entity['start']:entity['end']],
                'labels': [label]
            }
        })
    
    return results, unique_labels

# Load your CSV file
print("📂 Loading CSV file...")
csv_file = 'annotated_corpus_for_dataverse.csv'  # Update with your file path
df = pd.read_csv(csv_file)

# Filter for rows where relevance_label is not 'zero' (optional)
print("🔍 Filtering data...")
filtered_df = df[df['relevance_label'] ==2].copy()
print(f"✅ Filtered to {len(filtered_df)} rows")

# Prepare Label Studio tasks
all_entities = set()
tasks = []

print("\n📊 Processing texts and extracting entities...")

for index, row in filtered_df.iterrows():
    text = row['text_segment']
    article_id = row['id_segment']
    
    print(f"\nProcessing ID: {article_id}")
    print(f"Text preview: {text[:100]}...")
    
    # Extract entities using your model
    entities = ner_pipeline(text)
    
    # Convert to Label Studio format
    spans, unique_labels = ner_to_spans(text, entities)
    all_entities.update(unique_labels)
    
    # Create task with predictions
    task = {
        'data': {
            'text': text
        },
        'predictions': [{
            'model_version': 'ner_model_v1',
            'result': spans
        }]
    }
    
    tasks.append(task)
    
    # Print extracted entities for verification
    if spans:
        print(f"Found {len(spans)} entities:")
        for span in spans[:5]:  # Show first 5
            print(f"  - '{span['value']['text']}' → {span['value']['labels'][0]}")
        if len(spans) > 5:
            print(f"  ... and {len(spans) - 5} more")
    else:
        print("No entities found")

# Create output directory
output_dir = "label_studio_output"
os.makedirs(output_dir, exist_ok=True)

# Save tasks as Label Studio JSON format
output_file = os.path.join(output_dir, 'tasks.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(tasks, f, indent=2, ensure_ascii=False)

# Save entity labels
labels_file = os.path.join(output_dir, 'named_entities.txt')
with open(labels_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(sorted(all_entities)))

print(f"\n✅ Processing complete!")
print(f"📄 Tasks saved to: {output_file}")
print(f"📄 Entity labels saved to: {labels_file}")
print(f"📊 Total tasks: {len(tasks)}")
print(f"🏷️ Total unique entities: {len(all_entities)}")
print(f"🏷️ Entity types: {sorted(all_entities)}")

# XML Configuration for Label Studio
xml_config = """<View>
  <Text name="text" value="$text"/>
  <Labels name="label" toName="text">
"""
for label in sorted(all_entities):
    xml_config += f'    <Label value="{label}" background="yellow"/>\n'
xml_config += """  </Labels>
</View>"""

config_file = os.path.join(output_dir, 'label_studio_config.xml')
with open(config_file, 'w', encoding='utf-8') as f:
    f.write(xml_config)

print("\n📝 To import in Label Studio:")
print("1. Create a new project")
print(f"2. Go to Settings > Labeling Interface > Code")
print(f"3. Paste the content from: {config_file}")
print(f"4. Go to Import and upload: {output_file}")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


📂 Loading CSV file...
🔍 Filtering data...
✅ Filtered to 67 rows

📊 Processing texts and extracting entities...

Processing ID: 1-s2.0-S0303243414001718-main_19b
Text preview: #text': 'After the droughts in the 1970s and 1980s, the observed loss of woody vegetation cover was ...
Found 2 entities:
  - 'the 1970s and 1980s' → DATE
  - 'loss' → CHANGE

Processing ID: 1-s2.0-S0006320709005400-main_16b
Text preview: #text': 'The forests of West and Central Africa probably originally covered a combined area of about...
Found 5 entities:
  - 'forests' → LULC
  - 'West' → LOC
  - 'Central Africa' → LOC
  - 'area of about 3 million km' → SURFACE_UNIT
  - '2' → CARDINAL

Processing ID: 1-s2.0-S0140196322000143-main_121
Text preview: ).'}]}, {'@xmlns': 'http://www.tei-c.org/ns/1.0', 'head': {'@n': '4.2.', '#text': '[Land use ]change...
Found 9 entities:
  - '4.2' → CARDINAL
  - 'change' → CHANGE
  - '1996 to 2017' → DATE
  - 'increased' → CHANGE
  - '40% to 51.3%' → PERCENT
  ... and 4 more

Proce

In [59]:
def detailed_tokenization_check():
    """
    Deep dive into how tokenizer handles built-up area
    """
    print("\n🔧 DETAILED TOKENIZATION ANALYSIS:")
    print("=" * 50)
    
    test_sentence = "Agricultural land decreased by 30% while built-up area increased significantly SUB SAHARA."
    
    # Tokenize the full sentence
    encoding = tokenizer(
        test_sentence,
        return_offsets_mapping=True,
        add_special_tokens=True
    )
    
    tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    offsets = encoding['offset_mapping']
    
    print(f"📝 Full sentence: {test_sentence}")
    print(f"\n🔤 Tokenization:")
    
    for i, (token, (start, end)) in enumerate(zip(tokens, offsets)):
        text_span = test_sentence[start:end] if start != 0 or end != 0 else "[SPECIAL]"
        print(f"  {i:2d}: '{token}' → '{text_span}' (pos {start}-{end})")
        
        # Highlight built-up area tokens
        if 'built' in token.lower() or 'up' in token.lower() or 'area' in token.lower():
            print(f"      ⭐ BUILT-UP AREA RELATED TOKEN!")
    
    # Find where "built-up area" should be
    buildup_start = test_sentence.find("SUB SAHARA")
    buildup_end = buildup_start + len("SUB SAHARA")
    
    print(f"\n🎯 'built-up area' spans characters {buildup_start}-{buildup_end}")
    
    # Find which tokens cover this span
    covering_tokens = []
    for i, (start, end) in enumerate(offsets):
        if start != 0 or end != 0:  # Skip special tokens
            if not (end <= buildup_start or start >= buildup_end):  # Overlaps
                covering_tokens.append((i, tokens[i], start, end))
    
    print(f"🔍 Tokens covering 'built-up area':")
    for i, token, start, end in covering_tokens:
        print(f"  Token {i}: '{token}' (pos {start}-{end})")
    
    return covering_tokens

covering_tokens = detailed_tokenization_check()


🔧 DETAILED TOKENIZATION ANALYSIS:
📝 Full sentence: Agricultural land decreased by 30% while built-up area increased significantly SUB SAHARA.

🔤 Tokenization:
   0: '<s>' → '[SPECIAL]' (pos 0-0)
   1: 'Ag' → 'Ag' (pos 0-2)
   2: 'ric' → 'ric' (pos 2-5)
   3: 'ultural' → 'ultural' (pos 5-12)
   4: 'Ġland' → 'land' (pos 13-17)
   5: 'Ġdecreased' → 'decreased' (pos 18-27)
   6: 'Ġby' → 'by' (pos 28-30)
   7: 'Ġ30' → '30' (pos 31-33)
   8: '%' → '%' (pos 33-34)
   9: 'Ġwhile' → 'while' (pos 35-40)
  10: 'Ġbuilt' → 'built' (pos 41-46)
      ⭐ BUILT-UP AREA RELATED TOKEN!
  11: '-' → '-' (pos 46-47)
  12: 'up' → 'up' (pos 47-49)
      ⭐ BUILT-UP AREA RELATED TOKEN!
  13: 'Ġarea' → 'area' (pos 50-54)
      ⭐ BUILT-UP AREA RELATED TOKEN!
  14: 'Ġincreased' → 'increased' (pos 55-64)
  15: 'Ġsignificantly' → 'significantly' (pos 65-78)
  16: 'ĠSUB' → 'SUB' (pos 79-82)
  17: 'ĠSA' → 'SA' (pos 83-85)
  18: 'H' → 'H' (pos 85-86)
  19: 'ARA' → 'ARA' (pos 86-89)
  20: '.' → '.' (pos 89-90)
  21: '</

In [60]:
def check_buildup_labeling(found_examples):
    """
    Check if built-up area patterns are properly labeled as LULC
    """
    print("🔍 CHECKING IF BUILT-UP PATTERNS ARE LABELED AS LULC:")
    print("=" * 60)
    
    labeled_count = 0
    unlabeled_count = 0
    labeling_issues = []
    
    for example in found_examples[:20]:  # Check first 20 examples
        print(f"\n📝 Sentence: {example['sentence']}")
        print(f"🎯 Pattern: '{example['pattern']}'")
        
        if example['is_labeled_lulc']:
            print(f"✅ CORRECTLY LABELED as LULC")
            labeled_count += 1
        else:
            print(f"❌ NOT LABELED as LULC!")
            unlabeled_count += 1
            labeling_issues.append(example)
            
            # Show what entities ARE labeled in this sentence
            print("🏷️ Entities that ARE labeled:")
            for entity in example['all_entities']:
                print(f"   - '{entity['text']}' → {entity['label']}")
        
        print("-" * 40)
    
    print(f"\n📊 LABELING SUMMARY (from sample):")
    print(f"✅ Correctly labeled: {labeled_count}")
    print(f"❌ Not labeled: {unlabeled_count}")
    
    if unlabeled_count > 0:
        print(f"\n⚠️ WARNING: {unlabeled_count} examples have 'built-up area' in text but NOT labeled as LULC!")
        print("This could explain why your model doesn't recognize it.")
    
    return labeled_count, unlabeled_count, labeling_issues

# Run the labeling check
labeled_count, unlabeled_count, labeling_issues = check_buildup_labeling(found_examples)

🔍 CHECKING IF BUILT-UP PATTERNS ARE LABELED AS LULC:

📝 Sentence: The analysis of land use and land cover change presented above reveals mostly a decrease in vegetative areas and an increase built up areas agricultural land and water bodies
🎯 Pattern: 'built up area'
✅ CORRECTLY LABELED as LULC
----------------------------------------

📝 Sentence: The analysis of land use and land cover change presented above reveals mostly a decrease in vegetative areas and an increase built up areas agricultural land and water bodies
🎯 Pattern: 'built up areas'
✅ CORRECTLY LABELED as LULC
----------------------------------------

📝 Sentence: The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.
🎯 Pattern: 'built-up area'
✅ CORRECTLY LABELED as LULC
----------------------------------------

📝 Sentence: Under the business as usual scenario, prediction analysis for the year 2050 show that built u

In [77]:
# Debug Cell 1: Check what terms are actually loaded
print("=== VOCABULARY CHECK ===")
print(f"\nLULC vocabulary size: {len(lulc_vocab)}")
print("Sample LULC terms:")
for term in sorted(list(lulc_vocab))[:20]:  # Show first 20
    print(f"  - '{term}'")

print(f"\nPROCESS vocabulary size: {len(process_vocab)}")
print("Sample PROCESS terms:")
for term in sorted(list(process_vocab))[:20]:  # Show first 20
    print(f"  - '{term}'")

# Check if specific terms are in vocabularies
test_terms = ['bare ground', 'bare soil', 'built-up area', 'built up area', 'increase', 'decrease']
print("\nChecking specific terms:")
for term in test_terms:
    in_lulc = term.lower() in lulc_vocab
    in_process = term.lower() in process_vocab
    print(f"  '{term}': LULC={in_lulc}, PROCESS={in_process}")

=== VOCABULARY CHECK ===


NameError: name 'lulc_vocab' is not defined

In [50]:
def check_term_labeling_consistency(found_examples, search_term, expected_label, max_examples=20):
    
    print("=" * 70)
    
    labeled_count = 0
    unlabeled_count = 0    
    labeling_issues = []
    
    for example in found_examples[:max_examples]:
        print(f"\n📝 Sentence: {example['sentence']}")
        print(f"🎯 Pattern: '{example['pattern']}'")
        
        # Check if the pattern is labeled with the expected entity type
        is_labeled_correctly = any(
            expected_label.lower() in entity['label'].lower() and 
            example['pattern'].lower() in entity['text'].lower()
            for entity in example['all_entities']
        )
        
        if is_labeled_correctly:
            print(f"✅ CORRECTLY LABELED as {expected_label}")
            labeled_count += 1
        else:
            print(f"❌ NOT LABELED as {expected_label}!")
            unlabeled_count += 1
            labeling_issues.append(example)
            
            # Show what entities ARE labeled in this sentence
            print("🏷️ Entities that ARE labeled:")
            for entity in example['all_entities']:
                print(f"   - '{entity['text']}' → {entity['label']}")
        
        print("-" * 50)
    
    print(f"\n📊 LABELING SUMMARY for '{search_term}' → '{expected_label}':")
    print(f"✅ Correctly labeled: {labeled_count}")
    print(f"❌ Not labeled as {expected_label}: {unlabeled_count}")
    
    # Calculate consistency percentage
    total_checked = labeled_count + unlabeled_count
    if total_checked > 0:
        consistency_rate = (labeled_count / total_checked) * 100
        print(f"📈 Labeling consistency: {consistency_rate:.1f}%")
        
        if consistency_rate < 80:
            print(f"⚠️ WARNING: Low consistency rate! Only {consistency_rate:.1f}% of '{search_term}' examples are labeled as '{expected_label}'")
            print("This inconsistency could affect model performance.")
        elif consistency_rate == 100:
            print(f"🎉 PERFECT! All '{search_term}' examples are consistently labeled as '{expected_label}'!")
        else:            print(f"✅ Good consistency rate: {consistency_rate:.1f}%")
    
    return labeled_count, unlabeled_count, labeling_issues

def search_and_check_term_labeling(fixed_raw_data, search_patterns, expected_label, term_description="term"):
    """
    Complete function to search for patterns and check their labeling consistency
    
    Args:
        raw_data: Your training data
        search_patterns: List of regex patterns to search for
        expected_label: Expected entity label (e.g., 'LULC', 'LOC', 'CHANGE')
        term_description: Description of what you're searching for (for display)
    """
    print(f"🔍 SEARCHING AND CHECKING LABELING FOR: {term_description.upper()}")
    print("=" * 80)    
    # Search for patterns
    found_examples = []
    pattern_counts = {pattern: 0 for pattern in search_patterns}
    
    for item in fixed_raw_data:  # Use fixed_raw_data here
        text = item['original_sentence'].lower()
        entities = item['entities']
        
        for pattern in search_patterns:
            if pattern.lower() in text:
                pattern_counts[pattern] += 1
                
                # Check if pattern is labeled with expected entity type
                is_labeled_correctly = any(
                    pattern.lower() in entity['text'].lower() and 
                    entity['label'].lower() == expected_label.lower()  # Case-insensitive comparison
                    for entity in entities
                )
                
                found_examples.append({
                    'sentence': item['original_sentence'],
                    'pattern': pattern,
                    'is_labeled_correctly': is_labeled_correctly,
                    'all_entities': entities
                })
    
    # Display search results
    print(f"\n📊 SEARCH RESULTS for {term_description}:")
    print("-" * 50)
    for pattern, count in pattern_counts.items():
        print(f"'{pattern}': {count} occurrences")
    
    print(f"\nTotal examples found: {len(found_examples)}")
    
    if found_examples:
        # Check labeling consistency
        labeled_count, unlabeled_count, labeling_issues = check_term_labeling_consistency(
            found_examples, term_description, expected_label
        )
        
        return found_examples, pattern_counts, labeled_count, unlabeled_count, labeling_issues
    else:
        print(f"\n❌ NO EXAMPLES FOUND for {term_description}!")
        return [], pattern_counts, 0, 0, []

# Example usage functions for different entity types:

def check_lulc_terms(fixed_raw_data): # Pass fixed_raw_data
    """Check LULC term consistency"""
    lulc_patterns = [
        "built-up area", "built up area", "built-up areas",
        "forest area", "agricultural land", "urban areas",
        "commercial zones", "industrial parks", "green spaces",
        "residential areas", "wetlands", "grasslands"
    ]
    
    return search_and_check_term_labeling(
        fixed_raw_data, lulc_patterns, "LULC", "LULC terms"    )

def check_location_terms(fixed_raw_data):
    """Check LOC term consistency"""
    location_patterns = [
        "Sub-Saharan", "southern region", "Miombo ", "western region",
        "coastal areas", "rural areas", "mountain region", "central region"
    ]
    
    return search_and_check_term_labeling(
        fixed_raw_data, location_patterns, "LOC", "Location terms"
    )

def check_change_terms(fixed_raw_data):
    """Check CHANGE term consistency"""
    change_patterns = [
        "increased", "decreased", "expansion", "reduction",
        "conversion", "development", "urbanization", "deforestation"
    ]
    
    return search_and_check_term_labeling(
        fixed_raw_data, change_patterns, "CHANGE", "Change terms"
    )

def check_coordinate_terms(fixed_raw_data):
    """Check COORDINATES term consistency"""
    coord_patterns = [
        "23.5°N", "87.3°E", "coordinates", "latitude", "longitude"
    ]
    
    return search_and_check_term_labeling(
        fixed_raw_data, coord_patterns, "COORDINATES", "Coordinate terms"
    )

# Custom check function
def check_custom_terms(fixed_raw_data, patterns, expected_label, description):
    """
    Check any custom terms for any entity label
    
    Example usage:
    check_custom_terms(
        raw_data, 
        ["climate change", "global warming"],         "PROCESS", 
        "Climate terms"
    )
    """
    return search_and_check_term_labeling(
        fixed_raw_data, patterns, expected_label, description
    )

In [76]:
location_results = check_lulc_terms(raw_data)

🔍 SEARCHING AND CHECKING LABELING FOR: LULC TERMS

📊 SEARCH RESULTS for LULC terms:
--------------------------------------------------
'built-up area': 247 occurrences
'built up area': 8 occurrences
'built-up areas': 122 occurrences
'forest area': 82 occurrences
'agricultural land': 265 occurrences
'urban areas': 76 occurrences
'commercial zones': 0 occurrences
'industrial parks': 0 occurrences
'green spaces': 12 occurrences
'residential areas': 21 occurrences
'wetlands': 86 occurrences
'grasslands': 79 occurrences

Total examples found: 998

📝 Sentence: I doubt that this is for [agricultural land]
🎯 Pattern: 'agricultural land'
❌ NOT LABELED as LULC!
🏷️ Entities that ARE labeled:
--------------------------------------------------

📝 Sentence: However, conflicting hypotheses have been reported: on the one hand an expansion (into native [savanna]) and intensification (fertilizer) of cropland [cultivation], and on the other hand [agricultural land] abandonment, caused by, e.g., the civil